In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/S02_eda_inferencia_ab"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


In [ ]:
# === DATOS DE LA SESION EMBEBIDOS (para Google Colab) ======================
# Estos CSV no tienen mirror publico: viven solo en la carpeta data/ del curso.
# Viajan comprimidos DENTRO del cuaderno y se escriben antes de cualquier
# lectura, con los MISMOS bytes que en local (mismo SHA256): las cifras de la
# sesion no cambian. En local esta celda no hace nada.
import base64, gzip, os, sys

_EMBEBIDOS = {
    "sleep.csv":  # 238 B  sha256=5563ff4da6cf477d...
        "H4sIADFCmWoC/y2OOw7CMBBE+znLYnk2/sQ9DcdIEVFBUADB8ZmNaOzn0azf7tvnvtzWp63f177Ydd/eD7ucQcupG41wOzE1oWOyU04unFAiDSyokappFc2mVEQNXRTzHbN+mkUzhmXdA8wWk8ygzjTiBfrRc2k4KZVPGhbLBxbw73F52KxI5BKxW01V2EFptKlLxaFC4IDnYykP3w+d8M1b7gAAAA==",
}

def _escribir_embebidos(destinos=None):
    """Vuelca los CSV embebidos al directorio actual, a ./data y a ../data:
    los cuadernos del curso leen de uno u otro."""
    if destinos is None:
        destinos = [os.getcwd(), os.path.join(os.getcwd(), "data"),
                    os.path.join(os.path.dirname(os.getcwd()), "data")]
    escritos = []
    for _dest in destinos:
        try:
            os.makedirs(_dest, exist_ok=True)
        except OSError:
            continue
        for _nombre, _b64 in _EMBEBIDOS.items():
            _ruta = os.path.join(_dest, _nombre)
            if not os.path.exists(_ruta):
                with open(_ruta, "wb") as _fh:
                    _fh.write(gzip.decompress(base64.b64decode(_b64)))
                escritos.append(_nombre)
    return sorted(set(escritos))

if "google.colab" in sys.modules:
    _e = _escribir_embebidos()
    print("Datos de la sesion listos en Colab:", ", ".join(_e) if _e else "ya estaban")


# Sesión 2 — EDA + estadística inferencial + A/B testing

**Curso:** Herramientas para la Ciencia de Datos, Facultad de Negocios, UPC
**Programa:** Administración y Ciencia de Datos para Negocios
**Sesión 2** — Laboratorio de replicación en Google Colab

> **Cómo se abre este cuaderno.** El curso lo distribuye por **Google Drive**: en la
> carpeta compartida, clic derecho sobre el archivo → *Abrir con* → *Google
> Colaboratory*. Conviene empezar por **Archivo → Guardar una copia en Drive** para
> conservar el trabajo. No se requiere cuenta de GitHub ni instalar nada en el equipo:
> los datos de la sesión viajan dentro del propio cuaderno.
> **Carpeta del curso en Drive (Pregrado):** https://drive.google.com/drive/folders/1-YJxRt0n-UZwQCu03Lls2LGUYz6KMsl2


---

## Objetivos de aprendizaje

Al terminar esta sesión, quien la trabaje será capaz de:

- Realizar un **análisis exploratorio de datos (EDA)** visual y numérico de forma sistemática y documentar sus hallazgos.
- **Seleccionar y aplicar la prueba de hipótesis adecuada** según el tipo de variable y los supuestos, y construir **intervalos de confianza** clásicos y por *bootstrap*.
- **Diseñar un experimento A/B** —determinando tamaño muestral y poder—, analizar sus resultados y **distinguir la significancia estadística de la práctica** para comunicar una **recomendación de negocio**.

## 2. Mapa de la sesión: nueve capítulos en dos clases

La sesión ocupa **dos clases**. Cada capítulo lleva un código —2.1 a 2.9— que es **el mismo** en el sílabo, en la guía del docente, en la guía de laboratorio y en las diapositivas, de modo que se pueda pasar de un material a otro sin traducir numeraciones.

**JUEVES — 145 min de contenido** (bloque A1 de 75, receso de 15, bloque A2 de 70; antes, 20 min de control sobre la Sesión 1)

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **2.1** | ¿Por qué no corresponde modelar apenas se reciben los datos? | Sin celdas: se trabaja en clase (el atípico de 49 854 se muestra en la teoría) |
| **2.2** | ¿Qué es un análisis exploratorio y qué debe responder siempre? | «Teoría guiada» |
| **2.3** | ¿Cómo se visualiza aquello que los estadísticos no revelan? | «Teoría guiada» |
| **2.4** | ¿Cómo se determina si una diferencia es real o atribuible al azar? | «Teoría guiada» |
| **2.5** | ¿Se sostiene con datos reales? La réplica de Student (1908) | «Réplica de Student (1908)» — laboratorio, pasos 0 a 5 |

**VIERNES — 120 min corridos**

| Cód. | Pregunta que responde | Dónde vive en este cuaderno |
|---|---|---|
| **2.6** | ¿Cuándo se puede confiar en una prueba de hipótesis? | «Supuestos: cómo identificarlos y corregirlos» |
| **2.7** | ¿Cómo se verifica que el valor p no induce a error? | «Intervalos del A/B y *bootstrap* certificado» |
| **2.8** | ¿Qué decisión habilita? El experimento A/B de Cookie Cats | «Laboratorio de negocio: el A/B de *Cookie Cats*» — laboratorio, paso 6 |
| **2.9** | ¿Qué no se puede afirmar, y qué sigue en S03? | «Cierre» |

> El **control** de esta sesión se resuelve en aula, en la franja de 20 minutos del jueves siguiente, y cubre **los nueve capítulos**, de los dos días.


## Cómo leer este cuaderno

Este cuaderno no solo **se ejecuta**: explica cada paso. Convenciones:

- **❓ Qué se quiere averiguar** abre cada resultado importante: la pregunta que ese resultado contesta, qué decisión depende de ella y **qué significaría cada resultado posible, dicho antes de ver el número**. Conviene detenerse ahí y contestar mentalmente antes de ejecutar: un dato solo informa a quien traía una pregunta.
- **🔎 Qué hace este código** precede a cada celda de código; **📖 Cómo se lee esta salida** sigue a cada resultado numérico clave. **💡** añade intuición y **⚠️** marca un supuesto o alerta.
- El alumno **realiza la mecánica de forma manual** y la verifica contra la librería con `assert`: **🖐️ Cálculo manual**, **🧮 Matemática en el cuerpo** (derivación en LaTeX), **✅ Verificación desde la base** (recomputa el resultado y lo cruza con el Excel) y **🧱 Construcción del A/B desde cero**.
- **📄 En el paper** indica la procedencia exacta de cada resultado replicado (fuente, sección, tabla, página).
- La **«Sección 8» (Supuestos)** ejecuta los diagnósticos; su teoría vive en la guía de supuestos de la sesión. **Ninguna** de esas celdas escribe en el Excel de contrato.
- **Valor operativo vs. benchmark:** las cifras que se ejecutan aquí son las del **venv** (prevalecen); las de **Student (1908)** se citan como *benchmark* etiquetado y coinciden hasta la 4.ª cifra.
- **Convención Excel:** los resultados y pruebas se vuelcan a `resultados/S02_resultados.xlsx` y las **figuras de resultados se generan leyendo ese Excel**; las figuras de **EDA de datos crudos** y los **diagnósticos de supuestos** se trazan directamente de los datos (exención de EDA del estándar).

## Preparación del entorno (Sección 1 del cuaderno)

La siguiente celda instala, **solo en Google Colab**, las librerías con las versiones fijadas de la sesión. En Colab, `pandas`, `numpy`, `scipy`, `matplotlib` y `seaborn` suelen venir preinstalados; el pin de **`setuptools<81`** evita el fallo de `pkg_resources` que rompe la importación de `ydata-profiling`. En una ejecución local con el entorno del curso ya configurado, esta celda se omite (etiqueta `# SKIP-LOCAL`).

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys



**🔎 Qué hace este código.** La celda anterior instala **paquete por paquete** las versiones **certificadas** en la matriz de versiones certificada del curso (solo en Colab) con **degradación individual** —si un tag no resuelve, ese paquete se reintenta sin fijar y los demás no se ven afectados— y, en cualquier entorno, imprime la tabla **certificada → instalada** para que quede constancia de con qué versiones se produjo cada cifra. La celda siguiente importa las librerías de la sesión, fija la semilla del *bootstrap* (`rng`), configura el estilo de las figuras y **resuelve las rutas** (`data/`, `resultados/`, `figuras/`) tanto en ejecución local (nbconvert) como en Colab. `pingouin` es **opcional y NO está instalado en el venv del curso**: todas las cifras de este cuaderno se calculan con `scipy`/`statsmodels`, de modo que se ejecuta sin errores en cualquier caso.

In [ ]:
# Importaciones y configuración común
import sys, io
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
matplotlib.use("Agg")   # backend no interactivo: las figuras se guardan a disco
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower, TTestPower
from statsmodels.stats.multitest import multipletests

# pingouin es OPCIONAL y NO está en el venv del curso: todas las cifras se calculan con
# scipy/statsmodels; si estuviera disponible (Colab) solo se muestra su salida ampliada.
try:
    import pingouin as pg
    HAS_PINGOUIN = True
except Exception:
    HAS_PINGOUIN = False

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
rng = np.random.default_rng(42)   # semilla reproducible para el bootstrap

# --- Resolución de rutas: funciona en local (nbconvert) y en Colab ---
CWD = Path.cwd()
if (CWD / "data").exists():
    SESSION = CWD
elif (CWD.parent / "data").exists():
    SESSION = CWD.parent            # el cuaderno vive en notebook/
else:
    SESSION = CWD                   # Colab: todo en el directorio de trabajo
DATA_DIR = SESSION / "data"
RESULT_DIR = SESSION / "resultados"; RESULT_DIR.mkdir(exist_ok=True)
FIG_DIR = SESSION / "figuras"; FIG_DIR.mkdir(exist_ok=True)
XLSX = RESULT_DIR / "S02_resultados.xlsx"
print("Directorio de la sesión:", SESSION)
print(f"scipy {scipy.__version__} | pandas {pd.__version__} | "
      f"pingouin {'disponible' if HAS_PINGOUIN else 'no instalado (se usa scipy/statsmodels)'}")

## 2.2 a 2.4 — Teoría guiada: qué responde un EDA, cómo se visualiza y cómo se decide si una diferencia es real (Sección 2 del cuaderno)

Cada bloque presenta un concepto, una demostración ejecutable **corta** y su **lectura de negocio**. Las definiciones provienen de la investigación verificada de la sesión (el glosario de la sesión). No se ajusta ningún modelo de regresión: la correlación se **explora**, no se modela (eso corresponde a S03).

### 2.1 EDA: tipos de variable, descriptivos, atípicos y faltantes

El **análisis exploratorio de datos** examina un conjunto de datos —sobre todo de forma gráfica— antes de modelar, para descubrir estructura, anomalías y supuestos. El **tipo** de cada variable (cuantitativa vs. categórica) determina qué estadístico, gráfico y prueba son válidos. Los **atípicos** (fuera de $Q_1-1.5\cdot RIC$ y $Q_3+1.5\cdot RIC$) distorsionan medias y correlaciones; los **faltantes** (`NaN`) rara vez son aleatorios y sesgan conclusiones.

**🔎 Qué hace este código.** Sobre una tabla mínima con una variable continua (`ticket`) y una categórica (`canal`), muestra los **tipos**, los **descriptivos**, el **conteo de faltantes** y aplica la **regla del rango intercuartílico (RIC)** para marcar atípicos. Es el diagnóstico de EDA que precede a elegir cualquier prueba (la guía de supuestos de la sesión, Parte 3).

In [ ]:
# Mini-demo: descriptivos, tipos, atípicos (regla del RIC) y faltantes
demo = pd.DataFrame({
    "ticket":  [20.0, 22.5, 19.0, 21.0, 500.0, np.nan, 23.0, 18.5],  # continua (con atípico y faltante)
    "canal":   ["web", "web", "app", "app", "web", "app", "web", "app"],  # categórica nominal
})
print("Tipos de dato:\n", demo.dtypes, "\n")
print("Descriptivos de 'ticket':\n", demo["ticket"].describe(), "\n")
print("Faltantes por columna:\n", demo.isna().sum(), "\n")
q1, q3 = demo["ticket"].quantile([0.25, 0.75]); ric = q3 - q1
lim_inf, lim_sup = q1 - 1.5*ric, q3 + 1.5*ric
atipicos = demo.loc[(demo["ticket"] < lim_inf) | (demo["ticket"] > lim_sup), "ticket"]
print(f"Límites RIC: ({lim_inf:.2f}, {lim_sup:.2f}) -> atípicos: {atipicos.dropna().tolist()}")

**📖 Cómo se lee.** Un solo registro anómalo (un ticket de 500 frente a ~20) queda **fuera de los límites RIC** y desvía la media y cualquier KPI que la use; la **mediana** resiste mejor. **💡** Antes de decidir con una métrica conviene saber cuántos faltantes tiene y si su ausencia es sistemática (p. ej. los clientes de mayor renta que no declaran ingreso). El tratamiento de atípicos y faltantes se desarrolla en `SUPUESTOS_S02.md` (Parte 3.1).

### 2.2 Intervalo de confianza: clásico y por *bootstrap*

Un **IC 95%** es un rango de valores plausibles para un parámetro. El **clásico** de una media usa la distribución t; el de **bootstrap** remuestrea con reemplazo miles de veces y toma percentiles de la distribución empírica, sin exigir normalidad ni fórmula cerrada (Efron, 1979).

### 🧮 Matemática en el cuerpo — el IC 95 % de una media

Con una muestra de tamaño $n$, media $\bar{x}$ y desviación muestral $s$, el **error estándar** es $\mathrm{EE}=s/\sqrt{n}$ y el intervalo de confianza clásico al $95\%$ es

$$\bar{x} \;\pm\; t_{n-1,\,0.975}\cdot \frac{s}{\sqrt{n}},$$

donde $t_{n-1,\,0.975}$ es el cuantil de la distribución $t$ de Student con $n-1$ grados de libertad. **Lectura de cobertura (no de probabilidad del parámetro):** el $95\%$ **del procedimiento** significa que, en muestras repetidas, el $95\%$ de los intervalos así construidos contiene el valor verdadero — no que «hay $95\%$ de probabilidad de que el parámetro esté en este intervalo concreto» (`SUPUESTOS_S02.md`, Parte 1.5).

**🔎 Qué hace este código.** Calcula el IC 95 % de una media de **dos formas**: (a) el **clásico** con la fórmula $\bar{x}\pm t\cdot s/\sqrt{n}$ y (b) el de **bootstrap** (10 000 remuestreos con reemplazo y percentiles 2.5–97.5). Sirve para ver que ambos coinciden cuando los supuestos se cumplen.

In [ ]:
# Mini-demo: IC clásico vs. bootstrap para la media
muestra = np.array([12, 15, 14, 10, 18, 16, 11, 13, 17, 14, 12, 19], dtype=float)
n = muestra.size; media = muestra.mean(); ee = muestra.std(ddof=1)/np.sqrt(n)
tcrit = stats.t.ppf(0.975, n-1)
ic_clasico = (media - tcrit*ee, media + tcrit*ee)

B = 10_000
medias_boot = np.array([rng.choice(muestra, size=n, replace=True).mean() for _ in range(B)])
ic_boot = np.percentile(medias_boot, [2.5, 97.5])
print(f"Media = {media:.3f}")
print(f"IC 95% clásico   (t): ({ic_clasico[0]:.3f}, {ic_clasico[1]:.3f})")
print(f"IC 95% bootstrap    : ({ic_boot[0]:.3f}, {ic_boot[1]:.3f})")

**📖 Cómo se lee.** Los dos intervalos casi coinciden: el **clásico** da **(12.450, 16.050)** y el de ***bootstrap*** **(12.750, 15.750)** en torno a una media de **14.250**. Comunicar «la métrica está entre **~12.5 y ~16.0**» (o «~12.8 y ~15.8» con el *bootstrap*) es más riguroso que un número puntual, porque transmite la **incertidumbre**. **⚠️** El extremo del *bootstrap* depende de la semilla (con `rng` fijado en 42 se obtiene 15.750; con otras semillas 15.750–15.833), así que se reporta **junto con la semilla y el número de remuestreos**, nunca como una constante. **💡** Cuando la métrica no es una media simple (mediana, ratio, *lift*), el *bootstrap* da el intervalo sin necesidad de una fórmula cerrada — es exactamente lo que hace el **Anexo A** con la diferencia de proporciones del A/B.

### 2.3 Elegir la prueba: t de dos muestras, ANOVA y chi²

- **t de dos muestras** (independientes): compara la media de **dos grupos** de usuarios distintos (base del A/B con dos variantes). La variante de **Welch** (`equal_var=False`) es la recomendada por defecto.
- **ANOVA de un factor**: generaliza la t a **más de dos grupos** mediante el estadístico F.
- **chi² de independencia**: contrasta si dos variables **categóricas** están asociadas, sobre conteos (no medias).

**🔎 Qué hace este código.** Ejecuta las tres pruebas del árbol de decisión sobre datos mínimos: una **t de Welch** (2 grupos), un **ANOVA** (3 grupos) y un **chi²** de independencia (tabla de conteos). Cada una responde a un **tipo de dato y diseño** distinto.

In [ ]:
# t de dos muestras (Welch)
a = np.array([31, 35, 29, 40, 33, 36]); b = np.array([28, 30, 27, 33, 29, 31])
t_w, p_w = stats.ttest_ind(a, b, equal_var=False)
print(f"t de dos muestras (Welch): t={t_w:.3f}, p={p_w:.4f}")

# ANOVA de un factor (3 grupos)
g_A = [72, 75, 70, 74]; g_B = [78, 80, 77, 79]; g_C = [73, 71, 74, 72]
F, p_anova = stats.f_oneway(g_A, g_B, g_C)
print(f"ANOVA (F): F={F:.3f}, p={p_anova:.4f}")

# chi² de independencia (canal x conversión)
tabla = pd.DataFrame({"convierte": [120, 90], "no_convierte": [380, 410]}, index=["email", "redes"])
chi2, p_chi, dof, esp = stats.chi2_contingency(tabla, correction=False)
print(f"chi² de independencia: chi2={chi2:.3f}, gl={dof}, p={p_chi:.4f}")

**📖 Cómo se lee.** La prueba se elige por el **tipo de dato y el diseño**, no por costumbre: dos grupos → t; tres o más → ANOVA; proporciones o categorías → chi². El árbol de decisión de `plantillas/arbol_decision_prueba.docx` resume esta elección en una página; su fundamento está en `SUPUESTOS_S02.md` (Parte 3.3).

### 2.4 Supuestos: normalidad (Shapiro-Wilk) y alternativa no paramétrica (Mann-Whitney)

**Shapiro-Wilk** contrasta $H_0$: «los datos son normales»; un p pequeño la rechaza. Es una verificación de supuestos previa a las pruebas paramétricas y se acompaña de un histograma o Q-Q. Si la normalidad no se sostiene o la variable es muy sesgada, **Mann-Whitney U** compara dos grupos usando **rangos**, sin suponer una distribución concreta.

**🔎 Qué hace este código.** Genera dos muestras **sesgadas** (exponenciales), aplica **Shapiro-Wilk** sobre la primera (debe rechazar la normalidad) y las compara con **Mann-Whitney U** (alternativa no paramétrica basada en rangos).

In [ ]:
# Shapiro-Wilk sobre una muestra sesgada y su alternativa no paramétrica
sesgada_1 = rng.exponential(scale=2.0, size=40)
sesgada_2 = rng.exponential(scale=2.6, size=40)
W, p_sw = stats.shapiro(sesgada_1)
print(f"Shapiro-Wilk (grupo 1): W={W:.3f}, p={p_sw:.4f} -> "
      f"{'se rechaza' if p_sw < 0.05 else 'no se rechaza'} la normalidad")
U, p_mw = stats.mannwhitneyu(sesgada_1, sesgada_2, alternative="two-sided")
print(f"Mann-Whitney U: U={U:.1f}, p={p_mw:.4f}")

**📖 Cómo se lee.** Métricas como el gasto o la duración de sesión suelen ser muy sesgadas; aplicarles una t sin verificar la normalidad puede distorsionar la conclusión. **⚠️** Shapiro **no se usa solo**: se lee junto al histograma/Q-Q y a la luz del tamaño muestral (con n grande rechaza por desviaciones triviales; con n pequeño tiene poca potencia). Detalle en `SUPUESTOS_S02.md` (Parte 1.1).

### 2.5 Comparaciones múltiples: Bonferroni y FDR

Al evaluar **muchas** métricas o segmentos, alguna aparecerá «significativa» por azar. **Bonferroni** exige $p<\alpha/m$ (controla la probabilidad de *al menos un* falso positivo; conservador). **FDR de Benjamini-Hochberg** controla la *proporción* esperada de falsos positivos entre los hallazgos declarados y es más potente cuando hay muchas pruebas.

**🔎 Qué hace este código.** Toma 6 p-valores y aplica las dos correcciones (`multipletests`): **Bonferroni** (FWER) y **FDR de Benjamini-Hochberg**, mostrando qué hallazgos sobreviven a cada una.

In [ ]:
# Corrección de 6 p-valores por Bonferroni y por FDR (Benjamini-Hochberg)
pvals = np.array([0.001, 0.012, 0.030, 0.045, 0.190, 0.400])
rej_bonf, p_bonf, _, _ = multipletests(pvals, alpha=0.05, method="bonferroni")
rej_fdr,  p_fdr,  _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")
resumen = pd.DataFrame({"p_original": pvals,
                        "signif_Bonferroni": rej_bonf,
                        "signif_FDR_BH": rej_fdr})
print(resumen.to_string(index=False))

**📖 Cómo se lee.** Con Bonferroni sobreviven menos hallazgos que con FDR: un tablero con 40 métricas casi siempre mostrará 1–2 «ganadoras» espurias. **💡** Declarar de antemano la **métrica primaria (OEC)** y corregir por multiplicidad evita perseguir mejoras inexistentes (`SUPUESTOS_S02.md`, Parte 2.5).

### 2.6 A/B testing: hipótesis, tamaño/poder, p-valor, lift y *peeking*

Un **A/B test** asigna usuarios al azar a control (A) y tratamiento (B); la aleatorización permite atribución **causal** del efecto. Antes de lanzar se fija el **efecto mínimo relevante (MDE)** y se calcula el **tamaño muestral** para un **poder** de 0.8 con $\alpha=0.05$. El **p-valor** dice *si* hay efecto; el **lift** (cambio relativo) dice *cuánto*. Mirar el test repetidamente y pararlo al cruzar el umbral (**peeking**) infla los falsos positivos.

### 🧮 Matemática en el cuerpo — tamaño del efecto y tamaño muestral en proporciones

Para comparar dos proporciones $p_1$ y $p_2$, el **tamaño del efecto** de Cohen usa la transformación angular (estabiliza la varianza):

$$\varphi(p)=2\arcsin\!\sqrt{p},\qquad h=\varphi(p_2)-\varphi(p_1).$$

El **tamaño muestral por grupo** para un contraste bilateral con nivel $\alpha$ y potencia $1-\beta$ es, aproximadamente,

$$n \;=\; 2\left(\frac{z_{1-\alpha/2}+z_{1-\beta}}{h}\right)^{2},$$

de modo que **cuanto menor es el efecto $h$ que se quiere detectar, más grande es $n$** (crece como $1/h^2$). Con $\alpha=0.05$ y potencia $0.8$, $z_{0.975}=1.96$ y $z_{0.80}=0.84$. La **prueba z de dos proporciones** contrasta $H_0: p_1=p_2$ con

$$z=\frac{\hat p_2-\hat p_1}{\sqrt{\hat p\,(1-\hat p)\bigl(\tfrac{1}{n_1}+\tfrac{1}{n_2}\bigr)}},\qquad \hat p=\frac{x_1+x_2}{n_1+n_2}\ \text{(proporción combinada bajo }H_0\text{)}.$$

**🔎 Qué hace este código.** Traduce un **lift objetivo** (+5 % relativo) a un tamaño del efecto $h$ y resuelve el **tamaño muestral por grupo** para poder 0.8; luego contrasta un resultado ficticio con la **prueba z de dos proporciones** y calcula el lift observado.

In [ ]:
# Mini-demo A/B: tamaño muestral para un lift objetivo y p-valor de un contraste de proporciones
p_control = 0.20                      # conversión base
lift_obj = 0.05                       # se quiere detectar +5% relativo
p_trat = p_control * (1 + lift_obj)
h = proportion_effectsize(p_trat, p_control)   # tamaño del efecto (Cohen h)
n_grupo = NormalIndPower().solve_power(effect_size=h, alpha=0.05, power=0.8,
                                       ratio=1.0, alternative="two-sided")
print(f"Para detectar un lift de {lift_obj:.0%} (0.20 -> {p_trat:.3f}) con poder 0.8: "
      f"n ≈ {int(np.ceil(n_grupo)):,} por grupo")

# Contraste de proporciones con un resultado ficticio ya observado
exitos = np.array([232, 200]); n_obs = np.array([1000, 1000])
z, p_ab = proportions_ztest(exitos, n_obs)
lift_obs = (exitos[0]/n_obs[0]) / (exitos[1]/n_obs[1]) - 1
print(f"Resultado: z={z:.3f}, p={p_ab:.4f}, lift observado = {lift_obs:+.1%}")

**📖 Cómo se lee.** El tamaño muestral define cuánto tráfico y tiempo requiere el test; subdimensionarlo lo desperdicia. Con la duración fijada de antemano se evita el *peeking*. **💡** La decisión final combina **significancia** (p) y **relevancia práctica** (lift y su intervalo), no solo el p-valor (`SUPUESTOS_S02.md`, Partes 1.6 y 2.3).

## 2.5 — ¿Se sostiene con datos reales? La réplica de Student (1908) (Sección 3 del cuaderno)

> **Paso 0 — Contexto del paper.** **Student (1908)**, *The Probable Error of a Mean*, **Biometrika 6(1), 1–25**. «Student» es el seudónimo de **William Sealy Gosset**, químico-estadístico de la cervecería **Guinness** (Dublín), que necesitaba concluir a partir de **muestras pequeñas** y derivó la distribución hoy llamada **t de Student**. Como ejemplo reanalizó los datos de **Cushny & Peebles (1905)** sobre el efecto de dos hipnóticos en las **horas extra de sueño** de 10 pacientes: el dataset `sleep`.

> **⚠️ Advertencia de diseño (clave del laboratorio).** El CSV `sleep` está en formato largo con una columna `group` (fármaco 1 y 2), pero **no son dos grupos independientes**: son los **mismos 10 sujetos** (`ID` 1–10) medidos con **ambos fármacos**. El diseño es **PAREADO**. La réplica correcta usa una **prueba t pareada** (t≈4.06, p≈0.0028); aplicar por error una t de **dos muestras** da t≈1.86 y **no reproduce** el resultado publicado.

**Valor a reproducir (target declarado para la sesión «Sección 8»):** diferencia media = **1.58 h**, t pareada (gl=9) = **4.06**, p = **0.0028**, IC 95% ≈ **(0.70, 2.46)**, d_z de Cohen ≈ **1.28**.

### 3.0 Qué preguntaba el paper, y por qué usó lo que usó

**💡 Antes de trabajar con los datos.** Una réplica sin esta pregunta se convierte en mecánica: se ejecutan celdas y se obtiene un número. Lo que sigue explica **qué buscaba el autor** y **por qué eligió cada pieza de su método**, que es de donde procede el criterio para elegir un método propio mañana. *(Desarrollo completo con las citas del original: la ficha de la sesión de réplica del paper, «Sección 0».)*

**El objetivo no era comparar dos somníferos.** Student lo declara de forma explícita: «The aim of the present paper is to determine the point at which we may use the tables of the probability integral in judging of the significance of the mean of a series of experiments, and to furnish alternative tables for use when the number of experiments is too few» (p. 2) — *determinar el punto a partir del cual pueden usarse las tablas de la curva normal para juzgar la significación de una media, y proporcionar tablas alternativas cuando el número de experimentos es demasiado escaso*. La pregunta era **metodológica y de frontera**: ¿dónde termina la muestra «grande» y empieza la «pequeña», y qué se usa por debajo de ese límite? El fármaco ocupa media página al final del artículo —es la ilustración, no el objeto— y el estadístico que el autor define y tabula no se llama «t», sino **z**.

**Qué decisión dependía de la respuesta.** El destinatario es quien no puede repetir el experimento a voluntad: la muestra pequeña «itself affords the only indication of the variability» (p. 2). Y la conclusión IV del artículo enuncia el uso: juzgar si una serie de experimentos, por corta que sea, ya alcanza la exactitud requerida **o si es necesario continuar la investigación**. Es la regla para cerrar un experimento o prolongar la inversión en él — el mismo problema del analista que decide si detiene un test A/B o lo mantiene activo una semana más.

**Por qué no bastaba la normal con s/√n.** «As we decrease the number of experiments, the value of the standard deviation found from the sample of experiments becomes itself subject to an increasing error, until judgments reached in this way may become altogether misleading» (p. 1). El defecto no está en la media, sino en tratar una **s estimada como si fuera una σ conocida**. Con n = 10 la curva normal atribuye a un límite unas 7000 a 1 cuando las probabilidades reales rondan 550 a 1: «the normal curve gives a false feeling of security» (p. 13). La respuesta de Student consiste en dividir por la desviación de la **propia muestra** y aceptar que el denominador también es aleatorio, lo que obliga a derivar la distribución del cociente completo y produce las colas más gruesas de la t. La normalidad de la población se asume de forma **declarada y acotada** (p. 1) y se contrasta después contra distribuciones reales (Sección VI): supuesto declarado, puesto a prueba y limitado — el mismo orden que se sigue más abajo con Shapiro-Wilk antes de la prueba t.

**Por qué la serie de diferencias y no dos grupos.** Ante los dos fármacos, Student no compara dos columnas independientes: «This we must test by making a new series, subtracting 1 from 2» (Sección IX). Y justifica el resultado: «The low value of the S.D. is probably due to the different drugs reacting similarly on the same patient, so that there is correlation between the results» (Sección IX). En cifras del propio paper, la desviación estándar de las diferencias (**1,17**) es menor que la de cada columna por separado (**1,70** y **1,90**): el mismo paciente actúa como su propio control y el ruido entre personas desaparece del cálculo. El pareo **está en el artículo, con su justificación**; no es un tecnicismo moderno impuesto sobre datos antiguos.

**Por qué ese resultado responde la pregunta.** La Sección IX ofrece tres series. El fármaco 1 frente a nada da «about 8 to 1» (insuficiente); el fármaco 2 frente a nada, «nearly 400 to 1» (acredita al 2 como soporífero, pero no lo declara mejor que el 1); solo la diferencia pareada responde «¿cuál de los dos se receta?», con «about 666 to 1 that 2 is the better soporific». Es también la única serie que exhibe la ganancia del método: la desviación cae de 1,70-1,90 a 1,17 **por el pareo**, y el veredicto pasa de dudoso a firme sin añadir un solo paciente.

**⚠️ De ahí la diferencia que aparecerá al comparar.** Student informa **z = 1,35** y este cuaderno obtiene **t = 4,06**: es la misma evidencia en otras unidades. Su desviación usa divisor n y expresa la media en unidades de la desviación de la muestra, no del error estándar; la conversión es exacta, **t = z × √(n − 1)** = 1,354 × 3 = 4,06. Y una cautela que no se negocia: Student lee su tabla como «the chance that the mean of the population […] is positive» —probabilidad **sobre la hipótesis**—, mientras que el valor p moderno es probabilidad **sobre los datos** bajo H₀. Aquí p se reporta como p, nunca como «probabilidad de que el fármaco 2 sea mejor».


### 📄 En el paper — de dónde procede esta réplica

- **Paper seminal:** Student (1908), *The Probable Error of a Mean*, **Biometrika 6(1):1–25** (PDF del original: york.ac.uk/depts/maths/histstat/student.pdf).
- **Dato original:** Cushny & Peebles (1905), *The action of optical isomers*, reanalizado por Student; hoy es el dataset base `datasets::sleep` de R (mirror Rdatasets, 20 filas × 4 columnas). **Base original: sí.**
- **Valor que se reproduce:** el análisis **pareado** de las 10 diferencias intra-sujeto ($\bar d = 1.58$ h, t = 4.06, gl = 9, p = 0.0028).

Las 10 diferencias publicadas (fármaco 2 − fármaco 1):

| ID | Fármaco 1 | Fármaco 2 | Diferencia |
|---:|---:|---:|---:|
| 1 | 0.7 | 1.9 | 1.2 |
| 2 | −1.6 | 0.8 | 2.4 |
| 3 | −0.2 | 1.1 | 1.3 |
| 4 | −1.2 | 0.1 | 1.3 |
| 5 | −0.1 | −0.1 | 0.0 |
| 6 | 3.4 | 4.4 | 1.0 |
| 7 | 3.7 | 5.5 | 1.8 |
| 8 | 0.8 | 1.6 | 0.8 |
| 9 | 0.0 | 4.6 | 4.6 |
| 10 | 2.0 | 3.4 | 1.4 |

Detalle y verificación en la ficha de la sesión de réplica del paper.

### Paso 1 — Cargar la base original y verificar su forma

Se carga `sleep` desde `data/sleep.csv` (descargable con `data/descargar_datos.py`, fuente Rdatasets). Si el archivo no está disponible —por ejemplo en Colab sin la carpeta `data/`— se usa el **fallback embebido** con los valores **publicados** de Cushny-Peebles (no sintéticos). Se esperan **20 filas × 4 columnas** (`rownames`, `extra`, `group`, `ID`).

**🔎 Qué hace este código.** Define `cargar_sleep()`, que lee el CSV verificado o, si falta, reconstruye el dataframe con los **valores publicados**; luego verifica la forma (20 × 4) y muestra las primeras filas.

In [ ]:
def cargar_sleep() -> pd.DataFrame:
    ruta = DATA_DIR / "sleep.csv"
    if ruta.exists():
        df = pd.read_csv(ruta)
        print(f"sleep cargado de {ruta}")
        return df
    # Fallback: datos publicados (Cushny & Peebles, vía Student 1908), no sintéticos
    print("data/sleep.csv no encontrado; se usa el fallback embebido (datos publicados).")
    g1 = [0.7, -1.6, -0.2, -1.2, -0.1, 3.4, 3.7, 0.8, 0.0, 2.0]
    g2 = [1.9, 0.8, 1.1, 0.1, -0.1, 4.4, 5.5, 1.6, 4.6, 3.4]
    filas, idx = [], 1
    for g, vals in ((1, g1), (2, g2)):
        for sujeto, extra in enumerate(vals, start=1):
            filas.append({"rownames": idx, "extra": extra, "group": g, "ID": sujeto}); idx += 1
    return pd.DataFrame(filas)

sleep = cargar_sleep()
print("Forma:", sleep.shape, "| columnas:", list(sleep.columns))
sleep.head()

**📖 Cómo se lee.** La base tiene **20 filas × 4 columnas**: cada fila es una medición (`extra` = horas extra de sueño) de un sujeto (`ID`) bajo un fármaco (`group`). La forma confirma que se cargó la base correcta antes de analizar.

### Paso 2 — EDA breve y reconocimiento del diseño pareado

Se describe `extra` por grupo y se **comprueba que cada `ID` aparece en ambos grupos**: esa es la evidencia del diseño pareado.

**❓ Qué se quiere averiguar.** ¿Estos datos comparan **dos grupos de pacientes distintos**, o miden **dos veces a los mismos diez pacientes**?

- **Qué decide:** qué prueba es válida. De la respuesta depende la elección entre una t de dos muestras independientes y una t pareada — y esa misma pregunta se repetirá en todo experimento propio antes de aplicar una fórmula.
- **Antes de mirar el resultado:** si cada `ID` apareciese en un solo grupo, habría veinte sujetos y dos muestras independientes. Si cada `ID` aparece en **los dos** grupos, hay diez sujetos medidos dos veces: cada paciente es su propio control y la considerable variabilidad entre personas deja de introducir ruido en la comparación.

**🔎 Qué hace este código.** Calcula los descriptivos de `extra` por fármaco y **verifica que cada sujeto (`ID`) aparece en los dos grupos** — el rasgo distintivo del diseño pareado, que decide qué prueba es válida (`SUPUESTOS_S02.md`, Parte 1.3).

In [ ]:
print("Descriptivos de 'extra' por fármaco (group):")
descr_grupo = sleep.groupby("group")["extra"].agg(["count", "mean", "std"])
print(descr_grupo, "\n")

# Cada sujeto (ID) debe aparecer en los dos grupos -> diseño pareado
apariciones = sleep.groupby("ID")["group"].nunique()
print("¿Cada ID aparece en 2 grupos?", bool((apariciones == 2).all()),
      f"({sleep['ID'].nunique()} sujetos, {sleep['group'].nunique()} fármacos)")

**📖 Cómo se lee.** El fármaco 2 tiene media más alta (2.33 vs 0.75 h), pero lo decisivo es que **cada `ID` aparece en los dos grupos**: no son dos muestras independientes sino **medidas repetidas** del mismo sujeto. **⚠️** Ignorar esto y usar una t de dos muestras es el error deliberado que se demuestra en el Paso 6.

**🔎 Qué hace este código.** Traza, **de datos crudos** (exención de EDA del estándar), el boxplot y el histograma de `extra` por fármaco para juzgar tendencia central, dispersión y solape entre grupos.

In [ ]:
# Figura EDA (datos crudos): distribución de 'extra' por fármaco
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
sns.boxplot(data=sleep, x="group", y="extra", hue="group", palette="Set2",
            legend=False, ax=ax[0])
sns.stripplot(data=sleep, x="group", y="extra", color="black", size=5, alpha=0.6, ax=ax[0])
ax[0].set_title("Horas extra de sueño por fármaco (boxplot)")
ax[0].set_xlabel("Fármaco (group)"); ax[0].set_ylabel("extra (horas)")
for g, color in [(1, "#66c2a5"), (2, "#fc8d62")]:
    ax[1].hist(sleep.loc[sleep.group == g, "extra"], bins=6, alpha=0.6,
               label=f"Fármaco {g}", color=color, edgecolor="white")
ax[1].set_title("Distribución de 'extra' por fármaco (histograma)")
ax[1].set_xlabel("extra (horas)"); ax[1].set_ylabel("frecuencia"); ax[1].legend()
fig.tight_layout()
f_eda_sleep = FIG_DIR / "eda_sleep_extra_por_grupo.png"
fig.savefig(f_eda_sleep, dpi=150, bbox_inches="tight"); plt.close(fig)
display(Image(filename=str(f_eda_sleep)))

**📖 Cómo se lee.** El fármaco 2 tiende a producir más horas extra de sueño que el 1, pero hay mucha variabilidad **entre sujetos** (los boxplots se solapan). Justamente esa variabilidad entre personas es la que el diseño pareado neutraliza al trabajar las **diferencias**.

### Paso 3 — Preparar el pareo: formato ancho y diferencias intra-sujeto

Se pivota a una fila por sujeto (columnas fármaco 1 y 2) y se construye la **diferencia** $d = \text{extra}_{2} - \text{extra}_{1}$. Student trabajó sobre estas 10 diferencias.

**🔎 Qué hace este código.** Pivota `sleep` a **formato ancho** (una fila por sujeto), crea la columna `diferencia` = fármaco 2 − fármaco 1, y guarda los vectores `g1`, `g2` y `d` que usan todos los pasos siguientes.

In [ ]:
ancho = sleep.pivot(index="ID", columns="group", values="extra")
ancho.columns = ["farmaco_1", "farmaco_2"]
ancho["diferencia"] = ancho["farmaco_2"] - ancho["farmaco_1"]
g1 = ancho["farmaco_1"].to_numpy()
g2 = ancho["farmaco_2"].to_numpy()
d = ancho["diferencia"].to_numpy()
print(ancho, "\n")
print(f"Diferencia media = {d.mean():.4f}  |  desv. de las diferencias (ddof=1) = {d.std(ddof=1):.4f}")

**📖 Cómo se lee.** La diferencia media es **1.58 h** y la desviación de las diferencias **1.23 h**. Estas dos cifras bastan para reconstruir la prueba t pareada: el bloque siguiente lo realiza de forma manual.

### 🧮 Matemática en el cuerpo — el estadístico t pareado, su IC y el tamaño del efecto

La prueba t **pareada** equivale a una t de **una muestra** sobre las diferencias $d_i = x_{2i}-x_{1i}$. Con $\bar d$ (media de las diferencias), $s_d$ (su desviación muestral) y $n$ pares:

$$t=\frac{\bar d}{s_d/\sqrt{n}},\qquad \text{gl}=n-1.$$

El **intervalo de confianza** al $95\%$ de la diferencia media usa el mismo error estándar y el valor crítico $t$:

$$\bar d \;\pm\; t_{n-1,\,0.975}\cdot \frac{s_d}{\sqrt{n}}.$$

El **tamaño del efecto** apropiado para el diseño pareado es el $d_z$ de Cohen, que estandariza por la desviación de las **diferencias**:

$$d_z=\frac{\bar d}{s_d}.$$

**⚠️** No confundir con la $d$ «combinada» que estandariza por la desviación de **ambas condiciones** (como si fueran independientes): esa da $\approx 0.83$ y es la que reporta `pingouin` por defecto. Para el pareo el correcto es $d_z\approx 1.28$ (`REPLICACION_PAPER.md`, «Sección 5», nota del tamaño del efecto).

### 🖐️ Cálculo manual — la t pareada desde las 10 diferencias

Se reconstruye el estadístico paso a paso desde $\bar d$ y $s_d$, se calcula el p-valor y el IC 95 %, y se **verifica con `assert`** que reproduce exactamente `scipy.stats.ttest_rel` y el valor publicado por Student.

**❓ Qué se quiere averiguar.** ¿Cuánto más duerme un paciente con el fármaco 2 que con el fármaco 1, y esa diferencia se distingue del azar con solo **diez** pares?

- **Qué decide:** si la conclusión de Student (1908) se reproduce hoy sobre el mismo dato y, en el fondo, si una muestra pequeña autoriza a afirmar algo. La cervecería de Gosset no podía esperar a tener miles de observaciones; la mayoría de las decisiones de negocio, tampoco.
- **Antes de mirar el resultado:** la diferencia media debe coincidir con **1.58 h**, la cifra publicada por Student; si el cálculo manual se apartase de ella, la réplica fallaría y habría que revisar el pareo antes que la fórmula. Sobre el intervalo de 95 %: si incluyese el 0, la diferencia sería compatible con el azar; si queda entero por encima de 0, diez pares bastan para sostenerla.

**🔎 Qué hace este código.** Calcula $\bar d$, $s_d$, el error estándar, el estadístico $t=\bar d/\mathrm{EE}$, sus grados de libertad, el p-valor bilateral desde la distribución $t$, el IC 95 % y el $d_z$; luego los compara con `scipy` y con el **benchmark**: la **diferencia media 1.58 h es lo que publica Student (1908)**; **t = 4.0621, el p, el IC y el $d_z$ son cálculo propio** (`scipy`), verificado contra R `t.test(paired=TRUE)` en la ficha de la sesión de réplica del paper. Las dos columnas —lo publicado y lo calculado— **nunca se funden**.

In [ ]:
# 🖐️ La t pareada, reconstruida a mano desde las 10 diferencias, verificada contra scipy
n_d    = d.size
d_bar  = d.mean()                     # diferencia media
s_d    = d.std(ddof=1)                # desviación de las diferencias (muestral)
ee_d   = s_d / np.sqrt(n_d)           # error estándar de la media de diferencias
t_mano = d_bar / ee_d                 # estadístico t = d_barra / (s_d/raíz(n))
gl_mano = n_d - 1
p_mano = 2 * stats.t.sf(abs(t_mano), gl_mano)          # p bilateral desde la t de Student
t_crit = stats.t.ppf(0.975, gl_mano)                   # valor crítico t (gl=9)
ic_mano = (d_bar - t_crit*ee_d, d_bar + t_crit*ee_d)   # IC 95%
dz_mano = d_bar / s_d                                   # d_z de Cohen (pareado)
print(f"d_media = {d_bar:.4f} h   s_d = {s_d:.4f}   EE = {ee_d:.4f}   gl = {gl_mano}")
print(f"t = d_media/EE = {t_mano:.4f}   p = {p_mano:.6f}   "
      f"IC95 = ({ic_mano[0]:.4f}, {ic_mano[1]:.4f})   d_z = {dz_mano:.4f}")

# Verificación contra la librería y contra el benchmark.
# PROCEDENCIA (no se funden las columnas): d_media = 1.58 h es la cifra PUBLICADA por
# Student (1908) sobre los datos de Cushny-Peebles; t = 4.0621, gl = 9, p, IC y d_z son
# CÁLCULO PROPIO con scipy, verificado contra R `t.test(paired=TRUE)`
# (la ficha de réplica del paper).
ref = stats.ttest_rel(g2, g1)
assert abs(t_mano - float(ref.statistic)) < 1e-9, "la t a mano debe igualar a scipy.stats.ttest_rel"
assert abs(p_mano - float(ref.pvalue))    < 1e-9, "el p a mano debe igualar a scipy"
assert abs(d_bar - 1.58) < 0.01 and abs(t_mano - 4.0621) < 0.01 and gl_mano == 9
print("OK: la mecánica a mano reproduce scipy; d_media=1.58 h coincide con lo PUBLICADO por "
      "Student (1908) y t=4.0621/gl=9 con el cálculo propio verificado contra R.")

**📖 Cómo se lee.** La fórmula $t=\bar d/(s_d/\sqrt{n})$ da exactamente **4.0621** y el p-valor **0.002833**, idénticos a `scipy`: la prueba t pareada **no es una caja negra**, es esta división. El IC 95 % **(0.70, 2.46)** no incluye 0 y $d_z\approx1.28$ indica un efecto grande. Los `assert` fallarían si la mecánica no reprodujera la librería ni el benchmark: **1.58 h** es la cifra **publicada** por Student (1908); **4.0621**, el p, el IC y $d_z$ son **cálculo propio** verificado contra R.

### Paso 4 — Verificar supuestos: Shapiro-Wilk sobre las diferencias

La t pareada equivale a una t de una muestra sobre las diferencias, por lo que el supuesto de normalidad se verifica sobre $d$ (n=10). Si no se sostuviera, la alternativa no paramétrica sería la **prueba de Wilcoxon de rangos con signo**. El diagnóstico ampliado (Q-Q + Wilcoxon) está en la «Sección 8».

**❓ Qué se quiere averiguar.** ¿Se sostiene el supuesto de normalidad sobre el que descansa la t pareada que se acaba de calcular?

- **Qué decide:** si el p-valor y el intervalo del paso anterior se reportan tal cual, o si conviene acompañarlos de una alternativa no paramétrica antes de llevarlos a una decisión. El diagnóstico se realiza después porque las cifras se obtienen igual aunque el supuesto no se cumpla: ningún procedimiento emite una advertencia.
- **Antes de mirar el resultado:** con el umbral habitual, un **p ≥ 0,05** dejaría la normalidad sin objeción. Un **p < 0,05** la rechaza formalmente — y conviene recordar que con **n = 10** la prueba tiene poca potencia, de modo que el veredicto se lee junto al histograma y junto a la alternativa de rangos, nunca como sentencia aislada.

**🔎 Qué hace este código.** Ejecuta **Shapiro-Wilk** sobre las 10 diferencias y guarda `W_sw`, `p_sw` (se exportan a la hoja `supuestos` del Excel). Interpreta el resultado a la luz del tamaño muestral.

In [ ]:
W_sw, p_sw = stats.shapiro(d)
print(f"Shapiro-Wilk sobre las diferencias: W = {W_sw:.4f}, p = {p_sw:.4f}")
print("Interpretación:", "no se rechaza la normalidad (p >= 0.05)." if p_sw >= 0.05
      else "se rechaza la normalidad al 5% (p < 0.05); con n=10 conviene acompañar de\n"
           "un Q-Q plot y considerar Wilcoxon como alternativa. El resultado clásico de\n"
           "Student se mantiene como referencia histórica y didáctica.")

**📖 Cómo se lee.** Aquí Shapiro da **W = 0.83, p = 0.033 < 0.05**: con n = 10 la normalidad **se rechaza formalmente**. **⚠️** Esto **no invalida** la réplica: con muestras pequeñas Shapiro es poco potente y el resultado de Student es el histórico; además existe la alternativa no paramétrica (Wilcoxon). La lectura completa —Q-Q, Wilcoxon y el papel del TCL— se ejecuta en la «Sección 8» y se desarrolla en `SUPUESTOS_S02.md` (Parte 1.1).

### Paso 5 — Prueba t PAREADA (el resultado que reproduce el paper)

Se calcula con **`scipy.stats.ttest_rel`** (valores operativos que se exportan al Excel) y, si está disponible, se muestra la salida completa de **`pingouin.ttest(..., paired=True)`**. Se registran t, gl, p, IC 95% y el tamaño del efecto **d_z** (media de las diferencias / desviación de las diferencias), el apropiado para el diseño pareado.

**🔎 Qué hace este código.** Ejecuta la t pareada con `scipy` y guarda `dif_media, t_pareado, gl, p_valor, ic_inf, ic_sup, cohen_dz` (los que van al Excel). Calcula además la $d$ «combinada» ($\approx 0.83$) para explicar el número que muestra `pingouin`, y si `pingouin` está instalado imprime su tabla completa.

In [ ]:
# scipy: t, p e IC 95% de la diferencia (valores OPERATIVOS que se exportan al Excel)
res_sp = stats.ttest_rel(g2, g1)
ic = res_sp.confidence_interval(0.95)
dif_media = float(d.mean())
t_pareado = float(res_sp.statistic)
gl = int(len(d) - 1)
p_valor = float(res_sp.pvalue)
ic_inf, ic_sup = float(ic.low), float(ic.high)
cohen_dz = dif_media / float(d.std(ddof=1))   # d_z (pareado) = 1.58 / 1.2300

print("Resumen de la réplica (prueba t PAREADA):")
print(f"  diferencia media = {dif_media:.4f} h")
print(f"  t pareado (gl={gl}) = {t_pareado:.4f}")
print(f"  p-valor bilateral  = {p_valor:.6f}")
print(f"  IC 95%             = ({ic_inf:.4f}, {ic_sup:.4f})")
print(f"  d_z de Cohen       = {cohen_dz:.4f}")

# d de Cohen 'combinada' (desv. de ambas condiciones): es el ~0.83 que muestran por defecto
# las salidas ampliadas de la prueba t (p. ej. pingouin), pero aquí se CALCULA con numpy.
cohen_d_pool = dif_media / np.sqrt((g1.std(ddof=1)**2 + g2.std(ddof=1)**2) / 2)
poder_replica = TTestPower().power(effect_size=cohen_d_pool, nobs=len(d), alpha=0.05,
                                   alternative="two-sided")
print(f"\nNota: la d 'combinada' = {cohen_d_pool:.4f} (~0.83) trata las condiciones como "
      "independientes;\nel apropiado para el pareo es d_z ~ 1.28.")
print("\nSalida ampliada de la prueba t, CALCULADA CON scipy/statsmodels "
      "(pingouin NO está instalado en el venv del curso):")
print(f"  IC 95% de la diferencia = ({ic_inf:.4f}, {ic_sup:.4f})   [scipy.stats.ttest_rel]")
print(f"  d de Cohen (combinada)  = {cohen_d_pool:.4f}             [fórmula, numpy]")
print(f"  poder a posteriori      = {poder_replica:.4f}             [statsmodels TTestPower]")

# pingouin (si está disponible): salida completa T, dof, p-val, CI95%, cohen-d, power, BF10
if HAS_PINGOUIN:
    res_pg = pg.ttest(g2, g1, paired=True)
    print("\npingouin.ttest(paired=True):")
    print(res_pg.round(4).to_string())

**📖 Coincidencia con el benchmark.** Diferencia media **1.58 h**, t = **4.06**, p = **0.0028**, IC 95% **(0.70, 2.46)**, d_z ≈ **1.28**. **Procedencia exacta (las dos columnas no se funden):** la **diferencia media de 1.58 h** es la cifra **publicada** por Student (1908) a partir de los datos de Cushny & Peebles; **t, gl, p, IC y $d_z$ son cálculo propio** con `scipy`, **verificados contra R** `t.test(paired=TRUE)` (ver la ficha de la sesión de réplica del paper, «Sección 5»). El fármaco 2 añade en promedio 1.58 h de sueño frente al 1; como el IC **no incluye 0** y el efecto es grande (d_z≈1.28), la diferencia es estadística y prácticamente relevante. El **Anexo A.3** añade la **tercera vía** (*bootstrap* de las diferencias pareadas), que confirma el mismo signo sin suponer normalidad.

### Paso 6 — Contraste didáctico: la prueba EQUIVOCADA (dos muestras)

Se aplica deliberadamente una t de **dos muestras independientes**, que **ignora el pareo**. El resultado (t≈1.86, p≈0.079) **no alcanza significancia** y **no reproduce** el paper: la lección es que respetar el diseño pareado es lo que recupera el resultado y aumenta la potencia.

**❓ Qué se quiere averiguar.** ¿Cuánto cambia la conclusión —no el dato, la **conclusión**— cuando se elige la prueba equivocada para el diseño?

- **Qué decide:** hasta qué punto un acierto o un error metodológico, invisible en el informe final, determina lo que se afirma. Es el error que convierte un efecto real en un «no hay evidencia», y viceversa.
- **Antes de mirar el resultado:** el dato es exactamente el mismo que con la prueba pareada dio **t = 4.06** y **p = 0.0028**. Si la t de dos muestras devolviese cifras parecidas, el pareo sería un aspecto accesorio. Si el p sube **por encima de 0,05**, la misma evidencia deja de ser significativa solo por ignorar el diseño, y la decisión que se tome será la contraria.

**🔎 Qué hace este código.** Ejecuta `stats.ttest_ind(g2, g1)` (dos muestras, varianzas iguales) y guarda `t_ind`, `p_ind` (también van al Excel, como contraste), para compararlos con la t pareada correcta.

In [ ]:
res_ind = stats.ttest_ind(g2, g1)   # varianzas iguales (Student), como el contraste clásico
t_ind = float(res_ind.statistic); p_ind = float(res_ind.pvalue)
print(f"t de DOS MUESTRAS (independientes): t = {t_ind:.4f}, p = {p_ind:.4f}")
print(f"t PAREADA (correcta)             : t = {t_pareado:.4f}, p = {p_valor:.4f}")
print("\nLa prueba de dos muestras NO reproduce el resultado publicado: al tratar a los\n"
      "mismos sujetos como grupos distintos, la variabilidad entre personas 'ahoga' el\n"
      "efecto y el p-valor (0.079) ni siquiera cruza 0.05.")

# --- Figura: la prueba pareada reproduce a Student; la de dos muestras, no ---
# Comparación t pareada vs. t de dos muestras usando los estadísticos YA calculados
# (no lee ni escribe el Excel). El deck consume esta figura: figuras/grafico_t_pareado_vs_dos.png
t_crit_par = float(stats.t.ppf(0.975, gl))            # valor crítico t bilateral (gl = 9, pareado)
fig, ax = plt.subplots(figsize=(7.8, 3.9))
etiquetas = ["t PAREADA", "t DOS MUESTRAS"]
alturas = [t_pareado, t_ind]
notas_barra = ["correcta", "ignora el pareo"]
barras = ax.bar(etiquetas, alturas, color=["#c44e52", "#8c8c8c"], edgecolor="white", width=0.55)
ax.axhline(t_crit_par, ls="--", lw=1.4, color="#333333")
ax.text(0.5, t_crit_par + 0.12, f"umbral t≈{t_crit_par:.2f} (α=0.05)",
        ha="center", va="bottom", fontsize=9, color="#333333")
for b, pv, nota in zip(barras, [p_valor, p_ind], notas_barra):
    sig = "significativo" if pv < 0.05 else "no significativo"
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.12,
            f"t={b.get_height():.2f} — p={pv:.4f}\n({nota}: {sig})",
            ha="center", va="bottom", fontweight="bold", fontsize=9)
ax.set_ylabel("Estadístico t (valor absoluto)")
ax.set_title("Solo la t pareada reproduce a Student")
ax.set_ylim(0, max(alturas) + 0.95)
fig.tight_layout()
f_replica = FIG_DIR / "grafico_t_pareado_vs_dos.png"
fig.savefig(f_replica, dpi=150, bbox_inches="tight"); plt.close(fig)
display(Image(filename=str(f_replica)))

**📖 Cómo se lee.** El mismo dato da **t = 1.86, p = 0.079** (no significativo) si se ignora el pareo, frente a **t = 4.06, p = 0.0028** si se respeta. **💡** No es que una prueba «esté mal calculada»: es que **la prueba equivocada para el diseño** desperdicia la información del pareo. El árbol de decisión (`plantillas/arbol_decision_prueba.docx`) evita este error.

### Paso 7 — Interpretación de negocio

**Conclusión metodológica.** Cuando cada unidad es su propio control (mismo cliente antes/después, misma tienda con dos tratamientos), el **diseño pareado** elimina la variabilidad entre unidades y detecta efectos que un análisis de dos grupos independientes no vería. Elegir la prueba según el **diseño** —no por costumbre— es lo que separa una conclusión correcta de una equivocada, incluso con los mismos datos.

## 2.8 — ¿Qué decisión habilita? Laboratorio de negocio: el A/B de *Cookie Cats* (Sección 4 del cuaderno)

> **Del paper al negocio.** *Cookie Cats* (Tactile Entertainment) es un juego móvil de puzles. El experimento A/B mueve la primera «puerta» (*gate*) del **nivel 30** (control, `gate_30`) al **nivel 40** (tratamiento, `gate_40`) y mide el impacto en la **retención** de jugadores. La métrica de decisión es **`retention_7`** (volver a jugar 7 días después de instalar); `retention_1` es de diagnóstico. Datos: `data/cookie_cats.csv` (90 189 jugadores). La guía paso a paso está en `laboratorio/GUIA_LABORATORIO_S02.docx`.

### 📄 En el paper — origen del dato de Cookie Cats

- **Fuente del dato:** conjunto público del juego móvil **Cookie Cats** (Tactile Entertainment), popularizado como **caso de estudio de A/B testing** (mirror abierto `ryanschaub/Mobile-Games-A-B-Testing-with-Cookie-Cats`). **90 189** jugadores asignados al azar a `gate_30` (control) o `gate_40` (tratamiento).
- **No es un paper seminal**, sino el **caso de negocio** de la sesión: el marco de A/B testing que se aplica proviene de **Kohavi, Tang & Xu (2020)**, *Trustworthy Online Controlled Experiments* (aleatorización, potencia, SRM, *peeking*, guardarraíles).
- **Métrica primaria (OEC):** `retention_7`. Detalle en `data/diccionario_datos.md` y la bibliografía de la sesión.

**🔎 Qué hace este código.** Define `cargar_cookie()` (lee el CSV local o descarga el mirror abierto), carga la base y muestra su forma y el reparto por `version`.

In [ ]:
def cargar_cookie() -> pd.DataFrame:
    ruta = DATA_DIR / "cookie_cats.csv"
    if ruta.exists():
        df = pd.read_csv(ruta); print(f"cookie_cats cargado de {ruta}")
    else:
        url = ("https://raw.githubusercontent.com/ryanschaub/"
               "Mobile-Games-A-B-Testing-with-Cookie-Cats/master/cookie_cats.csv")
        print("data/cookie_cats.csv no encontrado; se descarga del mirror abierto...")
        df = pd.read_csv(url)
    return df

cc = cargar_cookie()
print("Forma:", cc.shape)
print(cc["version"].value_counts().to_string())
cc.head()

**📖 Cómo se lee.** La base tiene **90 189 filas × 5 columnas** y queda repartida en **44 700** (`gate_30`) y **45 489** (`gate_40`). Ese reparto casi 50/50 se auditará formalmente con la prueba de **SRM** en la «Sección 8» antes de confiar en el resultado.

### 4.1 EDA del engagement y la retención

`sum_gamerounds` (partidas en 14 días) es una métrica de *engagement* **muy sesgada** a la derecha, con un valor extremo (49 854 partidas). La retención se resume por grupo.

**🔎 Qué hace este código.** Describe `sum_gamerounds` (media, cuartiles, máximo) para exponer su asimetría, calcula las **tasas de retención** por grupo (`retention_1`, `retention_7`) y lanza, **solo si el entorno lo trae**, el **perfilado automático** de `ydata-profiling` (`ProfileReport`) sobre una muestra de 5 000 jugadores. **Decisión declarada:** `ydata-profiling` es la herramienta de perfilado que lista el sílabo, pero **no está instalada en el venv del curso**; por eso su uso es **opcional y degradable** —en Colab produce el informe interactivo; en local el cuaderno lo omite e imprime por qué—, y **ninguna cifra del material procede de él**: todas se calculan de forma explícita con pandas/scipy/statsmodels.


In [ ]:
print("Descriptivos de sum_gamerounds:")
print(cc["sum_gamerounds"].describe().to_string(), "\n")
print("Valor máximo (atípico extremo):", int(cc["sum_gamerounds"].max()), "\n")
print("Tasas de retención por grupo:")
print(cc.groupby("version")[["retention_1", "retention_7"]].mean().round(4).to_string())

# --- Perfilado automático OPCIONAL (ydata-profiling, herramienta del sílabo) ---
# DECISIÓN: no está en el venv del curso; se usa si el entorno lo trae (Colab) y se OMITE si no.
# Es un complemento exploratorio: ninguna cifra del material sale de aquí (ver la ficha de la sesión.
try:
    from ydata_profiling import ProfileReport
    _perfil = ProfileReport(cc.sample(n=5000, random_state=42),
                            title="Perfil EDA automático — Cookie Cats (muestra de 5 000)",
                            minimal=True, progress_bar=False)
    print("\nydata-profiling disponible: se muestra el perfil automático de la muestra.")
    _perfil.to_notebook_iframe()
except Exception as _e:
    print(f"\nydata-profiling NO disponible ({type(_e).__name__}): el perfilado automático se OMITE.")
    print("  Es un complemento OPCIONAL de Colab; el EDA de arriba (descriptivos, atípicos,"
          " faltantes y figuras) se calcula a mano y NO depende de esta librería.")


**📖 Cómo se lee.** La media de `sum_gamerounds` (~51) es mucho mayor que la mediana (~16) y el máximo es **49 854**: la métrica está **dominada por la cola** y su media resulta engañosa (conviene la mediana). Las tasas de retención bajan de `gate_30` a `gate_40` en ambas métricas; el contraste formal viene enseguida. **💡** Si el perfilado automático no aparece, es porque `ydata-profiling` no está instalado: es **deliberado**, el cuaderno omite ese paso de forma controlada y el EDA manual de esta sección cubre lo que la sesión evalúa.


**🔎 Qué hace este código.** Cubre el **Desarrollo 2 del sílabo** (visualización univariada y bivariada) con tres figuras trazadas **de datos crudos**: (1) el **histograma** de `sum_gamerounds` (recortado a ≤200 para ver la masa) junto al **heatmap de correlación** entre las métricas; (2) el **violín** de las partidas por variante —que muestra la forma completa de la distribución que el boxplot no muestra— junto a un **scatter bivariado** de partidas contra retención; y (3) el **pairplot**, la matriz de dispersión de todos los pares a la vez sobre una muestra de 2 000 jugadores. La correlación se **explora**, no se modela (eso es S03).


In [ ]:
# Figura EDA (datos crudos): histograma de sum_gamerounds (recortado) y heatmap de correlación
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
recorte = cc.loc[cc["sum_gamerounds"] <= 200, "sum_gamerounds"]
ax[0].hist(recorte, bins=50, color="#8da0cb", edgecolor="white")
ax[0].set_title("Partidas en 14 días (recortado a <=200)")
ax[0].set_xlabel("sum_gamerounds"); ax[0].set_ylabel("jugadores")
# Heatmap de correlación: la correlación se EXPLORA en EDA, no se modela (S03)
num = cc[["sum_gamerounds", "retention_1", "retention_7"]].astype(float)
corr = num.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, ax=ax[1])
ax[1].set_title("Correlación (exploratoria) entre métricas")
fig.tight_layout()
f_eda_cc = FIG_DIR / "eda_cookie_engagement_correlacion.png"
fig.savefig(f_eda_cc, dpi=150, bbox_inches="tight"); plt.close(fig)
display(Image(filename=str(f_eda_cc)))
print(f"Correlación máxima del heatmap (retention_1 ~ retention_7): r = {corr.loc['retention_1', 'retention_7']:.4f}")

# --- Desarrollo 2 del sílabo: VIOLÍN (univariada) + SCATTER BIVARIADO ---
cc_rec = cc.loc[cc["sum_gamerounds"] <= 200].copy()
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
sns.violinplot(data=cc_rec, x="version", y="sum_gamerounds", hue="version",
               palette=["#8da0cb", "#e78ac3"], cut=0, legend=False, ax=ax[0])
ax[0].set_title("Violín: la forma COMPLETA de las partidas por variante")
ax[0].set_xlabel(""); ax[0].set_ylabel("sum_gamerounds (recortado a <=200)")
# Scatter BIVARIADO: retention_7 es binaria (0/1), así que el par informativo no es
# jugador a jugador sino (partidas medianas del decil, tasa de retención del decil).
cc_d = cc.copy()
cc_d["decil"] = pd.qcut(cc_d["sum_gamerounds"].rank(method="first"), 10, labels=False)
puntos = (cc_d.groupby(["version", "decil"], observed=True)
              .agg(partidas=("sum_gamerounds", "median"), retencion=("retention_7", "mean"))
              .reset_index())
sns.scatterplot(data=puntos, x="partidas", y="retencion", hue="version", style="version",
                s=90, palette=["#8da0cb", "#e78ac3"], ax=ax[1])
ax[1].set_xscale("log")
ax[1].set_title("Scatter bivariado: partidas vs. retención a 7 días (por decil)")
ax[1].set_xlabel("sum_gamerounds — mediana del decil (escala log)")
ax[1].set_ylabel("tasa de retention_7")
fig.tight_layout()
f_eda_vs = FIG_DIR / "eda_cookie_violin_scatter.png"
fig.savefig(f_eda_vs, dpi=150, bbox_inches="tight"); plt.close(fig)
display(Image(filename=str(f_eda_vs)))

# --- PAIRPLOT: la matriz de dispersión de TODOS los pares de una mirada ---
muestra = cc_rec.sample(n=2000, random_state=42)[["sum_gamerounds", "retention_1",
                                                  "retention_7", "version"]]
gpair = sns.pairplot(muestra, hue="version", palette=["#8da0cb", "#e78ac3"], corner=True,
                     diag_kind="hist", plot_kws={"alpha": 0.35, "s": 14}, height=1.9)
gpair.figure.suptitle("Pairplot (muestra de 2 000 jugadores): todos los pares a la vez", y=1.02)
f_eda_pp = FIG_DIR / "eda_cookie_pairplot.png"
gpair.figure.savefig(f_eda_pp, dpi=140, bbox_inches="tight"); plt.close(gpair.figure)
display(Image(filename=str(f_eda_pp)))


**📖 Cómo se lee.** El *engagement* está dominado por muchos jugadores con pocas partidas y una cola muy extensa: la **media** de `sum_gamerounds` resulta engañosa y conviene la mediana. El **violín** añade lo que el boxplot no muestra —la masa se concentra cerca de 0 y la densidad desciende de forma abrupta—, y las dos variantes tienen **la misma forma**, que es precisamente lo que se espera de una asignación al azar. El **scatter bivariado** por deciles muestra que la retención a 7 días **crece con las partidas jugadas** y que las dos curvas se separan poco: el efecto del experimento es pequeño frente a la variación por *engagement*. El **pairplot** repite esa lectura para todos los pares a la vez. **⚠️** Todo esto es correlación **exploratoria** (la más alta del heatmap es **r = 0.33** entre `retention_1` y `retention_7`, fórmula en el glosario de la sesión sección E.35): asociación, no causa; no se ajusta ningún modelo sobre ella (`SUPUESTOS_S02.md`, Parte 3.3).


### 4.2 Contraste de proporciones: `retention_7` y `retention_1`

Se contrastan las proporciones de retención entre `gate_30` y `gate_40` con un **z-test de dos proporciones** (`statsmodels`), equivalente al **chi²** de independencia. Convención: diferencia y *lift* calculados como tratamiento (`gate_40`) menos/entre control (`gate_30`).

**❓ Qué se quiere averiguar.** ¿Mover la primera puerta del nivel 30 al nivel 40 **mejora o empeora** la retención de los jugadores?

- **La decisión concreta:** desplegar `gate_40` a toda la base o mantener `gate_30`. La métrica primaria declarada de antemano es `retention_7`; `retention_1` es complementaria, pero no determina la decisión.
- **Antes de mirar el resultado:** si la retención de `gate_40` sube y el p queda por debajo de 0,05, el cambio se defiende. Si **baja** con p pequeño, el cambio daña la métrica de decisión y no se despliega. Y si la diferencia no alcanza significancia, el experimento no autoriza ningún cambio: habrá que preguntarse si tenía tamaño suficiente, que es precisamente lo que mide el bloque siguiente.

**🔎 Qué hace este código.** Define `contraste_retencion(metrica)`, que cuenta éxitos y tamaños por grupo, calcula tasas, diferencia, lift y el **z de dos proporciones**; lo aplica a `retention_1` y `retention_7` y construye el DataFrame `ab_retencion` (se exporta al Excel).

In [ ]:
def contraste_retencion(metrica: str) -> dict:
    g = cc.groupby("version")[metrica].agg(["sum", "count"])
    s30, n30 = int(g.loc["gate_30", "sum"]), int(g.loc["gate_30", "count"])
    s40, n40 = int(g.loc["gate_40", "sum"]), int(g.loc["gate_40", "count"])
    p30, p40 = s30/n30, s40/n40
    z, pval = proportions_ztest([s40, s30], [n40, n30])   # tratamiento vs. control
    diff = p40 - p30
    lift = diff / p30
    return {"metrica": metrica, "gate30_exitos": s30, "gate30_n": n30, "gate30_tasa": p30,
            "gate40_exitos": s40, "gate40_n": n40, "gate40_tasa": p40,
            "diferencia": diff, "lift_pct": lift*100, "z_stat": float(z), "p_valor": float(pval)}

ab_rows = [contraste_retencion("retention_1"), contraste_retencion("retention_7")]
ab_retencion = pd.DataFrame(ab_rows)
print(ab_retencion.round(5).to_string(index=False))
print("\nretention_7: gate_40 = {:.4f} vs gate_30 = {:.4f} -> lift {:+.2f}% (p = {:.5f})".format(
    ab_rows[1]["gate40_tasa"], ab_rows[1]["gate30_tasa"], ab_rows[1]["lift_pct"], ab_rows[1]["p_valor"]))

**📖 Cómo se lee.** En `retention_7` la retención **cae** de 19.02% (`gate_30`) a 18.20% (`gate_40`): un *lift* de **−4.31%**, estadísticamente significativo (z = −3.16, p≈0.0016). En `retention_1` la caída (−1.32%) **no** es significativa (z = −1.78, p≈0.074). Mover la puerta al nivel 40 **empeora** la retención de largo plazo. El z se reconstruye de forma manual en el bloque siguiente y el **efecto con su intervalo** (IC de la diferencia y del *lift*, y su *bootstrap*) se calcula y registra en el **Anexo A**.

### 🖐️ Cálculo manual — el z de dos proporciones (retention_7)

Se reconstruye el estadístico $z$ desde los conteos por grupo (proporción combinada bajo $H_0$) y se **verifica con `assert`** que reproduce `statsmodels.proportions_ztest` y el valor del contrato (z = −3.164, p = 0.0016).

**🔎 Qué hace este código.** Cuenta éxitos y tamaños de `retention_7` por grupo, calcula la **proporción combinada** $\hat p$, el error estándar bajo $H_0$, el estadístico $z=(\hat p_B-\hat p_A)/\mathrm{EE}$ y su p-valor bilateral; luego los compara con la librería y con el contrato.

In [ ]:
# 🖐️ El z de dos proporciones (retention_7), reconstruido a mano y verificado contra statsmodels
g7 = cc.groupby("version")["retention_7"].agg(["sum", "count"])
x_ctrl, n_ctrl = int(g7.loc["gate_30", "sum"]), int(g7.loc["gate_30", "count"])   # control
x_trat, n_trat = int(g7.loc["gate_40", "sum"]), int(g7.loc["gate_40", "count"])   # tratamiento
p_ctrl, p_trat = x_ctrl/n_ctrl, x_trat/n_trat
p_comb = (x_ctrl + x_trat) / (n_ctrl + n_trat)                     # proporción combinada bajo H0
ee = np.sqrt(p_comb*(1-p_comb)*(1/n_ctrl + 1/n_trat))             # error estándar bajo H0
z_mano = (p_trat - p_ctrl) / ee
p_mano = 2 * stats.norm.sf(abs(z_mano))
print(f"p_control = {p_ctrl:.4f}   p_tratamiento = {p_trat:.4f}   p_combinada = {p_comb:.4f}")
print(f"z = (p_B - p_A)/EE = {z_mano:.4f}   p = {p_mano:.6f}")

z_lib, p_lib = proportions_ztest([x_trat, x_ctrl], [n_trat, n_ctrl])
assert abs(z_mano - float(z_lib)) < 1e-9, "el z a mano debe igualar a statsmodels.proportions_ztest"
assert abs(p_mano - float(p_lib)) < 1e-9
assert abs(z_mano - (-3.164)) < 0.01 and abs(p_mano - 0.0016) < 0.0005
print("OK: la mecánica a mano reproduce statsmodels y el contrato (z=-3.164, p=0.0016).")

**📖 Cómo se lee.** El z de dos proporciones **no es una caja negra**: es la diferencia de tasas dividida por su error estándar bajo $H_0$, y da **−3.164** (p = 0.00155), idéntico a `statsmodels`. El signo negativo confirma que el tratamiento (`gate_40`) **reduce** la retención. Los `assert` anclan el resultado al contrato del Excel.

### 4.3 Tamaño muestral y poder

Se calcula el **tamaño muestral por grupo** necesario para detectar, con poder 0.8 y $\alpha=0.05$, un *lift* objetivo del **2%** sobre la retención base de `gate_30` (`retention_7`), y el **poder observado** dado el efecto real y el n del experimento.

**❓ Qué se quiere averiguar.** ¿Cuánto tráfico habría hecho falta para detectar una mejora del **2 %** en la retención, y el experimento tenía capacidad real para ver el efecto que encontró?

- **Qué decide:** si un test se puede lanzar con el tráfico disponible y cuánto debe durar. Un test subdimensionado consume usuarios y semanas sin llegar a una conclusión, y su «no hay diferencia» no significa que no la haya.
- **Antes de mirar el resultado:** cuanto más pequeño el efecto que se quiere detectar, más observaciones exige, y la relación no es proporcional: crece mucho más deprisa. Si el n requerido para ese 2 % quedase por debajo de los ~45 000 jugadores por rama que tuvo el experimento, el diseño tenía margen suficiente; si lo supera, ese lift pequeño era indetectable desde el inicio, con independencia de la calidad del diseño.

**🔎 Qué hace este código.** Traduce un lift objetivo del 2 % a un tamaño del efecto $h$ y resuelve el **n por grupo** para poder 0.8; luego calcula el **poder observado** con el efecto real y los tamaños del experimento. Guarda `p_base, lift_objetivo, p_mde, h_mde, n_requerido, poder_obs` (van al Excel).

In [ ]:
analysis = NormalIndPower()
p_base = float(cc.loc[cc.version == "gate_30", "retention_7"].mean())   # ~0.1902
lift_objetivo = 0.02
p_mde = p_base * (1 + lift_objetivo)
h_mde = proportion_effectsize(p_mde, p_base)
n_requerido = analysis.solve_power(effect_size=abs(h_mde), alpha=0.05, power=0.8,
                                   ratio=1.0, alternative="two-sided")

# Poder observado con el efecto real de retention_7 y el n del experimento
p40_7 = float(cc.loc[cc.version == "gate_40", "retention_7"].mean())
h_obs = proportion_effectsize(p_base, p40_7)
n30 = int((cc.version == "gate_30").sum()); n40 = int((cc.version == "gate_40").sum())
poder_obs = analysis.solve_power(effect_size=abs(h_obs), nobs1=n30, alpha=0.05,
                                 ratio=n40/n30, alternative="two-sided")
n_requerido = float(np.ceil(n_requerido))
print(f"Retención base (gate_30, retention_7): {p_base:.4f}")
print(f"Para un lift objetivo de {lift_objetivo:.0%} -> tasa {p_mde:.4f}, "
      f"tamaño del efecto (h) = {abs(h_mde):.5f}")
print(f"Tamaño muestral requerido (poder 0.8, alfa 0.05): {int(n_requerido):,} por grupo")
print(f"Poder observado (efecto real de retention_7, n={n30:,}/{n40:,}): {poder_obs:.4f}")

**📖 Cómo se lee.** Detectar un *lift* pequeño (2%) exigiría **~168 000 jugadores por grupo**: los efectos pequeños requieren un tráfico considerable. El **poder observado (~0.89)** que se acaba de calcular es el del efecto de `retention_7`, **no** el de `retention_1`; además el poder *post-hoc* es una **reexpresión monótona del p-valor** (a menor p, mayor poder), de modo que es **señal, no veredicto** y no demuestra que la no-significancia de `retention_1` «no se deba a falta de potencia». Para juzgar si `retention_1` quedó **subdimensionado** se compara su poder **a priori** frente a un **efecto mínimo relevante (MDE)** de negocio —cuánto tráfico habría hecho falta para detectarlo—, nunca el poder del efecto ya observado. **💡** El cálculo **ex-ante** (no post-hoc) es la defensa contra el *peeking*; la **calculadora reutilizable** está en `plantillas/calculadora_tamano_muestral_ab.ipynb` (`SUPUESTOS_S02.md`, Parte 2.3).

### 4.4 Riesgo de *peeking*, significancia práctica y recomendación de negocio

- **Peeking.** Revisar el test a diario y detenerlo al ver p<0.05 habría inflado los falsos positivos: aquí la duración y el tamaño se fijan de antemano.
- **Significancia estadística vs. práctica.** El *lift* de −4.31% en `retention_7` es **significativo y relevante**: no es una diferencia trivial inflada por el tamaño muestral, sino una caída material en la métrica de decisión.

> **Recomendación de negocio.** **Mantener la puerta en el nivel 30 (`gate_30`).** Mover la puerta al nivel 40 **reduce** la retención a 7 días de forma estadística y prácticamente significativa (−4.31%, p≈0.0016), sin mejorar la retención a 1 día. Para dimensionar el daño en jugadores: esa caída de **0,82 pp** de `retention_7`, a título **ilustrativo** sobre la base del experimento (90 189 jugadores), equivale a ≈ **740 jugadores** que no vuelven al día 7 (cifra ilustrativa; no se estiman ARPU ni ingresos). Como `retention_7` es la métrica primaria (OEC) y actúa como guardarraíl del *engagement* de largo plazo, no se justifica el cambio. Esta recomendación, con su hipótesis, tamaño/poder, p-valor, *lift* y riesgo de *peeking*, es la base del entregable evaluable (`evaluacion/entregable.docx`).

## Transversal — Exportación a Excel y figuras de resultados (Sección 5 del cuaderno)

Por convención del curso, **todos los resultados y pruebas** se vuelcan a `resultados/S02_resultados.xlsx` (una hoja por bloque) y las **figuras de resultados se generan LEYENDO ese Excel**, nunca desde los objetos en memoria. (Las figuras de **EDA de datos crudos** de las secciones 3 y 4 sí se trazan directamente de los datos, por la exención de EDA de los estándares.) El contrato exacto de celdas de la hoja `prueba_t_pareada` lo audita el material de referencia de la sesión.

**🔎 Qué hace este código.** Construye las **5 hojas de contrato** del Excel a partir de los valores ya calculados (réplica t, descriptivos, supuesto de normalidad, contraste A/B y potencia) y las escribe con `openpyxl`. Es la **única** celda que escribe las 5 hojas de contrato; su lógica no debe cambiarse. (El **Anexo A** añade después **tres hojas nuevas** —`ab_intervalos`, `bootstrap_ab`, `bootstrap_sleep`— en modo *append*, y verifica con `assert` que estas 5 quedan intactas.)

In [ ]:
# --- Hojas de resultados ---
# 1) prueba_t_pareada: contrato exacto de celdas A1:B10 (valores CALCULADOS arriba)
prueba_t = pd.DataFrame({
    "metrica": ["diferencia_media", "t_pareado", "gl", "p_valor", "ic95_inf",
                "ic95_sup", "cohen_dz", "t_dos_muestras", "p_dos_muestras"],
    "valor":   [dif_media, t_pareado, gl, p_valor, ic_inf, ic_sup, cohen_dz, t_ind, p_ind],
})

# 2) descriptivos: media y desv. de extra por grupo
descriptivos = (sleep.groupby("group")["extra"].agg(["mean", "std"])
                .reset_index().rename(columns={"group": "grupo", "mean": "media", "std": "desv"}))

# 3) supuestos: Shapiro-Wilk (W, p) sobre las diferencias
supuestos = pd.DataFrame({"prueba": ["shapiro_wilk_diferencias"],
                          "W": [float(W_sw)], "p_valor": [float(p_sw)]})

# 4) ab_retencion: conteos, tasas, diferencia, lift, z y p por métrica
#    (ya calculado en 'ab_retencion')

# 5) ab_potencia: tamaño requerido para lift objetivo y poder observado
ab_potencia = pd.DataFrame({
    "concepto": ["retencion_base_gate30_ret7", "lift_objetivo_relativo",
                 "tasa_mde_objetivo", "effect_size_h", "alfa", "poder_objetivo",
                 "n_por_grupo_requerido", "poder_observado_ret7"],
    "valor":    [p_base, lift_objetivo, p_mde, abs(h_mde), 0.05, 0.80,
                 n_requerido, float(poder_obs)],
})

with pd.ExcelWriter(XLSX, engine="openpyxl") as xw:
    prueba_t.to_excel(xw, sheet_name="prueba_t_pareada", index=False)      # A1='metrica' B1='valor'
    descriptivos.to_excel(xw, sheet_name="descriptivos", index=False)
    supuestos.to_excel(xw, sheet_name="supuestos", index=False)
    ab_retencion.to_excel(xw, sheet_name="ab_retencion", index=False)
    ab_potencia.to_excel(xw, sheet_name="ab_potencia", index=False)

print("Excel escrito en:", XLSX)
print("\nHoja prueba_t_pareada (contrato A1:B10):")
print(prueba_t.to_string(index=False))

**📖 Cómo se lee.** El Excel queda con 5 hojas de contrato; la hoja `prueba_t_pareada` es el **contrato** que valida el material de referencia de la sesión (celdas **B2..B10**: diferencia 1.58, t 4.0621, gl 9, p 0.0028, IC 0.7001/2.4599, d_z 1.2846, t dos muestras 1.8608 y **p dos muestras 0.0792** — las nueve, B10 incluida). A partir de aquí, **toda figura de resultados se relee de este archivo**, no de memoria. El **Anexo A** añadirá tres hojas más (`ab_intervalos`, `bootstrap_ab`, `bootstrap_sleep`), también certificadas por el validador, **sin modificar** estas cinco.

### 5.1 Figuras de resultados generadas LEYENDO el Excel

Se **releen** las hojas del `.xlsx` y se trazan las figuras de resultados: la comparación de `retention_7` entre grupos y las tasas de ambas retenciones. Ninguna cifra proviene de memoria.

**🔎 Qué hace este código.** Relee la hoja `ab_retencion` del Excel y traza dos figuras de resultados: (1) `retention_7` `gate_30` vs `gate_40` con su lift y p, y (2) las tasas de ambas retenciones por grupo. Todas las cifras provienen del Excel.

In [ ]:
# Releer desde el Excel (no desde memoria) y graficar
ab_xl = pd.read_excel(XLSX, sheet_name="ab_retencion")

# Figura 1: retention_7 gate_30 vs gate_40 (métrica de decisión)
r7 = ab_xl.loc[ab_xl["metrica"] == "retention_7"].iloc[0]
fig, ax = plt.subplots(figsize=(6.2, 4.4))
barras = ax.bar(["gate_30\n(control)", "gate_40\n(tratamiento)"],
                [r7["gate30_tasa"]*100, r7["gate40_tasa"]*100],
                color=["#4c72b0", "#c44e52"], edgecolor="white")
ax.set_ylabel("Retención día 7 (%)")
ax.set_title(f"Retención a 7 días, lift {r7['lift_pct']:+.2f}%  (p = {r7['p_valor']:.4f})")
for b in barras:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.15, f"{b.get_height():.2f}%",
            ha="center", va="bottom", fontweight="bold")
ax.set_ylim(0, max(r7["gate30_tasa"], r7["gate40_tasa"])*100 + 3)
fig.tight_layout()
f_r7 = FIG_DIR / "resultado_ab_retention7.png"
fig.savefig(f_r7, dpi=150, bbox_inches="tight"); plt.close(fig)
display(Image(filename=str(f_r7)))

# Figura 2: retention_1 y retention_7 por grupo (barras agrupadas)
x = np.arange(len(ab_xl)); w = 0.38
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.bar(x - w/2, ab_xl["gate30_tasa"]*100, w, label="gate_30 (control)", color="#4c72b0")
ax.bar(x + w/2, ab_xl["gate40_tasa"]*100, w, label="gate_40 (tratamiento)", color="#c44e52")
ax.set_xticks(x); ax.set_xticklabels(ab_xl["metrica"])
ax.set_ylabel("Retención (%)"); ax.set_title("Tasas de retención por grupo (desde el Excel)")
ax.legend()
fig.tight_layout()
f_ret = FIG_DIR / "resultado_ab_retenciones.png"
fig.savefig(f_ret, dpi=150, bbox_inches="tight"); plt.close(fig)
display(Image(filename=str(f_ret)))
print("Figuras de resultados guardadas en:", FIG_DIR)

**📖 Cómo se lee.** Las barras muestran la caída de `retention_7` (19.02% → 18.20%) y el patrón paralelo en `retention_1`. Como se generan **releyendo el Excel**, si alguien editara una celda del contrato la figura cambiaría en consecuencia: dato y gráfico quedan acoplados.

## Transversal — Verificación desde la base (Sección 6 del cuaderno) <a id="verif-base"></a>

Antes de que el deck y la evaluación confíen en el Excel, se comprueba que ese registro es **producto de ejecutar el código sobre la base**, no un valor tecleado. Se **recomputa** —de forma independiente de los objetos ya calculados— la t pareada desde `sleep` y el z de `retention_7` desde `cookie_cats`, y se cruza con el Excel mediante `assert`. Refleja, visible y explicada, la lógica de el material de referencia de la sesión.

**🔎 Qué hace este código.** Recarga `sleep` y `cookie_cats` desde cero, recomputa la t pareada (t, p, d_z) y el z de `retention_7`, lee las celdas del Excel y comprueba con `assert` que **recomputado ≈ paper ≈ Excel**. No escribe en el Excel.

In [ ]:
# Recomputar t pareada (sleep) y A/B (cookie_cats) desde la base y cruzar con el Excel (NO escribe)
from openpyxl import load_workbook

# (1) t pareada recomputada desde sleep, independiente de los objetos previos
s_raw = cargar_sleep()
w_raw = s_raw.pivot(index="ID", columns="group", values="extra")
g1_v, g2_v = w_raw[1].to_numpy(), w_raw[2].to_numpy()
d_v = g2_v - g1_v
r_v = stats.ttest_rel(g2_v, g1_v)
t_v, p_v = float(r_v.statistic), float(r_v.pvalue)
dz_v = d_v.mean() / d_v.std(ddof=1)

# (2) A/B retention_7 recomputado desde cookie_cats
cc_raw = pd.read_csv(DATA_DIR / "cookie_cats.csv") if (DATA_DIR / "cookie_cats.csv").exists() else cc
gr = cc_raw.groupby("version")["retention_7"].agg(["sum", "count"])
s30v, n30v = int(gr.loc["gate_30", "sum"]), int(gr.loc["gate_30", "count"])
s40v, n40v = int(gr.loc["gate_40", "sum"]), int(gr.loc["gate_40", "count"])
z_v, pz_v = proportions_ztest([s40v, s30v], [n40v, n30v])

# (3) Leer el Excel de contrato
wb = load_workbook(XLSX, data_only=True)
pt = wb["prueba_t_pareada"]; xls_t = {pt[f"A{r}"].value: pt[f"B{r}"].value for r in range(2, 11)}
ab = pd.read_excel(XLSX, sheet_name="ab_retencion"); r7 = ab.loc[ab["metrica"] == "retention_7"].iloc[0]

tabla = pd.DataFrame({
    "recomputado (venv)": [round(d_v.mean(), 4), round(t_v, 4), round(p_v, 6),
                            round(dz_v, 4), round(float(z_v), 4), round(float(pz_v), 6)],
    "paper / referencia": [1.58, 4.0621, 0.0028, 1.28, -3.164, 0.0016],
    "Excel (contrato)":   [round(xls_t["diferencia_media"], 4), round(xls_t["t_pareado"], 4),
                            round(xls_t["p_valor"], 6), round(xls_t["cohen_dz"], 4),
                            round(float(r7["z_stat"]), 4), round(float(r7["p_valor"]), 6)],
}, index=["dif_media", "t_pareado", "p_valor", "d_z", "z_ab_ret7", "p_ab_ret7"])
print(tabla.to_string())

assert abs(t_v - xls_t["t_pareado"]) < 1e-6 and abs(p_v - xls_t["p_valor"]) < 1e-6
assert abs(d_v.mean() - xls_t["diferencia_media"]) < 1e-6 and abs(dz_v - xls_t["cohen_dz"]) < 1e-6
assert abs(float(z_v) - float(r7["z_stat"])) < 1e-6 and abs(float(pz_v) - float(r7["p_valor"])) < 1e-6
assert abs(t_v - 4.0621) < 0.15 and abs(p_v - 0.0028) < 0.001        # dentro de tolerancia del paper
assert abs(float(z_v) - (-3.164)) < 0.01
print("\nOK: recomputado desde la base ~ paper (tolerancia) y ~ Excel (contrato) — registro auditable.")

**📖 Cómo se lee.** Las tres columnas coinciden: lo **recomputado** desde `sleep` y `cookie_cats` reproduce el **paper** (t 4.06, p 0.0028; z −3.16) y coincide con el **Excel** hasta el 6.º decimal. Los `assert` fallarían si alguien editara el Excel manualmente o si la base cambiara: por eso el registro es **auditable**. La versión ejecutable con red vive en el material de referencia de la sesión.

## 2.4 en profundidad — Construcción del A/B test desde cero (Sección 7 del cuaderno)  <a id="sec-ab"></a>

El alumno **reconstruye el experimento sin la función auxiliar**: del CSV crudo a la decisión, pasando por conteos por grupo, prueba z de dos proporciones y lectura. El objetivo es reproducir, de forma independiente, el **mismo resultado del contrato** (z = −3.164, p = 0.0016) y llegar a la recomendación. Si coincide (con `assert`), queda demostrado que **es el método —no un atajo— el que produce la decisión**.

**🔎 Qué hace este código.** Lee `cookie_cats.csv` desde cero, construye la **tabla de conteos** por grupo para `retention_7`, calcula de forma manual la prueba z de dos proporciones, deriva el lift y la **decisión**, y comprueba con `assert` contra la librería y el Excel. No usa `contraste_retencion` ni objetos previos; tampoco reescribe el Excel.

In [ ]:
# A/B test reconstruido desde el CSV crudo, sin la función auxiliar (NO escribe en el Excel)
from openpyxl import load_workbook

raw = pd.read_csv(DATA_DIR / "cookie_cats.csv") if (DATA_DIR / "cookie_cats.csv").exists() else cc.copy()

# 1) Conteos por grupo para la métrica de decisión (retention_7)
tab = raw.groupby("version")["retention_7"].agg(exitos="sum", n="count")
exito_A, n_A = int(tab.loc["gate_30", "exitos"]), int(tab.loc["gate_30", "n"])   # control
exito_B, n_B = int(tab.loc["gate_40", "exitos"]), int(tab.loc["gate_40", "n"])   # tratamiento
pA, pB = exito_A/n_A, exito_B/n_B

# 2) Prueba z de dos proporciones a mano (varianza combinada bajo H0)
p_pool = (exito_A + exito_B) / (n_A + n_B)
se = np.sqrt(p_pool*(1-p_pool)*(1/n_A + 1/n_B))
z = (pB - pA) / se
p_val = 2*stats.norm.sf(abs(z))
lift = (pB - pA) / pA
print(f"control  gate_30: {exito_A:,}/{n_A:,} = {pA:.4f}")
print(f"tratam.  gate_40: {exito_B:,}/{n_B:,} = {pB:.4f}")
print(f"z = {z:.4f}   p = {p_val:.6f}   lift = {lift:+.2%}")

# 3) Decisión + verificación contra la librería y el Excel de contrato
z_lib, p_lib = proportions_ztest([exito_B, exito_A], [n_B, n_A])
ab = pd.read_excel(XLSX, sheet_name="ab_retencion"); r7 = ab.loc[ab["metrica"] == "retention_7"].iloc[0]
assert abs(z - float(z_lib)) < 1e-9 and abs(p_val - float(p_lib)) < 1e-9
assert abs(z - float(r7["z_stat"])) < 1e-6 and abs(p_val - float(r7["p_valor"])) < 1e-6
assert abs(z - (-3.164)) < 0.01 and p_val < 0.05
decision = "NO lanzar gate_40 (mantener gate_30)" if (p_val < 0.05 and lift < 0) else "evaluar lanzamiento"
print(f"\nDecisión: {decision} — la caída de retención a 7 días es significativa (p={p_val:.4f}).")
print("OK: el A/B armado desde cero reproduce el contrato (z=-3.164, p=0.0016).")

**📖 Cómo se lee.** Sin la función auxiliar, los mismos pasos (conteos → z → lift → decisión) devuelven **z = −3.164, p = 0.0016** y la recomendación de **mantener `gate_30`**. La coincidencia con el Excel (verificada por `assert`) demuestra que la decisión de negocio es reproducible desde el dato crudo, no un artefacto del código de conveniencia.

## 2.6 — ¿Cuándo se puede confiar en una prueba de hipótesis? Supuestos: cómo identificarlos y corregirlos (Sección 8 del cuaderno) <a id="supuestos"></a>

> **Fuente canónica:** la guía de supuestos de la sesión (qué es, cómo se identifica, cómo se corrige por método, alcance). Aquí se ejecutan los **diagnósticos**; su desarrollo teórico y sus fuentes viven en ese documento. **Ninguna celda de esta sección escribe en el Excel de contrato.**
>
> **Regla de alcance de S02.** El trabajo sobre cada supuesto llega hasta **(a) diagnosticarlo** —con su prueba, gráfico o señal y un umbral práctico— y **(b) aplicar o nombrar la corrección propia de la sesión** (elegir la prueba según el diseño, Welch por defecto, alternativa no paramétrica/bootstrap, dimensionar ex-ante, aplicar la verificación de SRM, corregir por multiplicidad y leer bien el IC y el valor p). Lo que pertenece a otras sesiones (supuestos del **OLS → S03**; inferencia causal e interferencia → **S12**; series → **S11**) o a un tratamiento más fino se **nombra**, no se ejecuta.

**Inferencia paramétrica (prueba t, IC, valor p)** — `SUPUESTOS_S02.md`, Parte 1:

| Supuesto | Cómo identificar | Cómo corregir (método) | Alcance |
|---|---|---|---|
| **1.1 Normalidad (y TCL)** | Q-Q + Shapiro-Wilk (p<0,05); histograma; regla del TCL (n≥30–40/grupo) | No paramétrica (Wilcoxon/Mann-Whitney); bootstrap; transformar; apoyarse en el TCL | **S02** |
| **1.2 Igualdad de varianzas** | Boxplot; Levene/Brown-Forsythe (p<0,05); cociente de desviaciones | **Welch por defecto** (`equal_var=False`); Mann-Whitney; transformar | **S02 — Welch por defecto** |
| **1.3 Independencia** | Revisar el diseño; en `sleep`, cada `ID` en ambos grupos ⇒ pareado | Elegir **t pareada** vs. independiente; definir la unidad experimental | **S02 — central** |
| **1.5–1.6 Lectura del IC y del valor p** | ¿cobertura o «prob. del parámetro»?; p = P(datos\|H₀); significancia ≠ relevancia | Reportar efecto + IC; fijar α/hipótesis antes; distinguir signif. estadística vs. práctica | **S02 — central** |

**A/B testing y EDA** — `SUPUESTOS_S02.md`, Partes 2 y 3:

| Supuesto | Cómo identificar | Cómo corregir (método) | Alcance |
|---|---|---|---|
| **2.1 Asignación aleatoria** | ¿aleatorizador o autoselección?; balance/A-A | Aleatorizar la unidad correcta; A/A previo — *cuasi-experimentos → S12* | **S02 — central** |
| **2.3 Tamaño/potencia** | Cálculo ex-ante (MDE, α, potencia) con `solve_power` | Dimensionar antes de lanzar; MDE de negocio | **S02 — ex-ante** |
| **2.4 SRM** | **Chi-cuadrado sobre el conteo** vs. razón esperada; SRM si p<0,001 | Investigar el **pipeline** (causa raíz); descartar el resultado hasta explicarlo | **S02** |
| **2.5 Multiplicidad / peeking** | Contar métricas/segmentos; ¿n y duración fijados antes? | **Bonferroni** / **FDR**; declarar OEC; cerrar duración | **S02** |
| **3.1–3.3 EDA (atípicos, forma, tipo)** | Boxplot/RIC; histograma/violín; clasificar tipo | Mediana/robustos; transformar; el EDA **alimenta el árbol de decisión** | **S02** |

**🔎 Qué hace este código (Diagnóstico 1.1 — normalidad).** Sobre las 10 diferencias de `sleep`, aplica **Shapiro-Wilk**, dibuja el **Q-Q plot** y el histograma, y ejecuta la alternativa no paramétrica **Wilcoxon de rangos con signo**. Umbral: p<0,05 rechaza normalidad — leído junto al tamaño muestral.

In [ ]:
# Diagnóstico 1.1 - normalidad de las diferencias: Shapiro-Wilk + Q-Q + Wilcoxon (NO escribe en el Excel)
W_d, p_d = stats.shapiro(d)
print(f"Shapiro-Wilk sobre las {d.size} diferencias: W = {W_d:.4f}, p = {p_d:.4f} -> "
      f"{'se rechaza' if p_d < 0.05 else 'no se rechaza'} la normalidad al 5%")
w_stat, w_p = stats.wilcoxon(g2, g1)   # alternativa no paramétrica (rangos con signo)
print(f"Wilcoxon de rangos con signo (alternativa): estadístico = {w_stat:.1f}, p = {w_p:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
stats.probplot(d, dist="norm", plot=ax[0])
ax[0].set_title("Q-Q de las diferencias (normalidad)")
ax[1].hist(d, bins=6, color="#8da0cb", edgecolor="white")
ax[1].set_title("Histograma de las diferencias"); ax[1].set_xlabel("d = extra_2 - extra_1")
fig.tight_layout()
f_qq = FIG_DIR / "S02_sup_qq_diferencias.png"
fig.savefig(f_qq, dpi=150, bbox_inches="tight"); plt.close(fig)
display(Image(filename=str(f_qq)))

**📖 Cómo se lee.** Shapiro rechaza la normalidad (**W = 0.83, p = 0.033**) y en el Q-Q los puntos se apartan de la recta en las colas — esperable con **n = 10**. **⚠️** Dos matices salvan la conclusión: (1) el **TCL** vuelve robusta a la t con n grande, y (2) la alternativa **Wilcoxon** también da **p = 0.004** (significativa), de modo que el efecto persiste sin suponer normalidad; y (3) el ***bootstrap*** de la media de las diferencias pareadas (**Anexo A.3**, B = 10 000, semilla 42) da un IC que **no incluye 0**. Son las **tres vías** —t pareada, Wilcoxon y *bootstrap*— y las tres coinciden. Con muestras pequeñas, Shapiro es poco potente y se lee junto al Q-Q (`SUPUESTOS_S02.md`, Parte 1.1).

**🔎 Qué hace este código (Diagnóstico 1.2 — igualdad de varianzas).** Aplica **Levene/Brown-Forsythe** sobre las dos condiciones de `sleep`, calcula el cociente de desviaciones y compara **Student vs. Welch** para la t de dos muestras. (Se comparan como independientes **solo para ilustrar el diagnóstico de varianzas**; el análisis correcto de `sleep` sigue siendo el pareado del Paso 5.)

In [ ]:
# Diagnóstico 1.2 - igualdad de varianzas entre condiciones: Levene -> Welch por defecto (NO escribe)
stat_lev, p_lev = stats.levene(g1, g2, center="median")   # Brown-Forsythe (robusta a asimetría)
sd1, sd2 = g1.std(ddof=1), g2.std(ddof=1)
print(f"Levene/Brown-Forsythe (H0: varianzas iguales): estadístico = {stat_lev:.4f}, p = {p_lev:.4f}")
print(f"Desv. estándar: fármaco 1 = {sd1:.4f} | fármaco 2 = {sd2:.4f} | "
      f"cociente = {max(sd1, sd2)/min(sd1, sd2):.2f}")
t_st, p_st = stats.ttest_ind(g2, g1, equal_var=True)     # Student (asume varianzas iguales)
t_we, p_we = stats.ttest_ind(g2, g1, equal_var=False)    # Welch (recomendado por defecto)
print(f"t de dos muestras - Student: t = {t_st:.4f}, p = {p_st:.4f} | "
      f"Welch: t = {t_we:.4f}, p = {p_we:.4f}")

**📖 Cómo se lee.** Levene **no rechaza** la igualdad de varianzas (p = 0.62) y el cociente de desviaciones (~1.1) está cerca de 1: aquí Student y Welch dan casi lo mismo (t ≈ 1.86). **💡** Aun así, la recomendación moderna es **usar Welch por defecto** (`equal_var=False`): controla mejor el error de tipo I cuando las varianzas difieren y pierde muy poca potencia cuando son iguales, evitando encadenar «Levene → decide qué t» (`SUPUESTOS_S02.md`, Parte 1.2).

**❓ Qué se quiere averiguar.** ¿La asignación aleatoria funcionó, o el reparto entre control y tratamiento está desbalanceado e invalida toda la comparación?

- **Qué decide:** si el resultado del A/B es fiable. Un desbalance en el reparto delata un fallo de instrumentación —un filtro, un bot, un despliegue parcial— y obliga a **descartar** el análisis en vez de interpretarlo, con independencia del p-valor del efecto.
- **Antes de mirar el resultado:** el diseño repartía 50/50 y se observaron **44 700 frente a 45 489**. Con el umbral estricto de SRM (**p < 0,001**), un p por debajo declara desajuste y detiene el análisis; un p por encima lo deja continuar. Nótese por qué el umbral no es 0,05: con 90 189 jugadores, un desbalance trivial produce p pequeños y una alarma falsa.

**🔎 Qué hace este código (Diagnóstico 2.4 — SRM).** Toma los **conteos por grupo** del experimento (44 700 vs 45 489) y aplica un **chi-cuadrado de bondad de ajuste** contra la razón de diseño 50/50, con el **umbral estricto** de SRM (p<0,001).

In [ ]:
# Diagnóstico 2.4 - SRM (Sample Ratio Mismatch): chi-cuadrado sobre los conteos de grupo (NO escribe)
ab = pd.read_excel(XLSX, sheet_name="ab_retencion")
n_A = int(ab.loc[ab["metrica"] == "retention_7", "gate30_n"].iloc[0])
n_B = int(ab.loc[ab["metrica"] == "retention_7", "gate40_n"].iloc[0])
obs = np.array([n_A, n_B]); total = obs.sum(); esp = np.array([total/2, total/2])   # razón de diseño 50/50
chi2_srm, p_srm = stats.chisquare(f_obs=obs, f_exp=esp)
print(f"Conteos observados: gate_30 = {n_A:,} ({n_A/total:.4%}) | gate_40 = {n_B:,} ({n_B/total:.4%})")
print(f"chi² (gl=1) = {chi2_srm:.4f}, p = {p_srm:.6f}")
umbral = 0.001
print(f"Umbral estricto de SRM (Kohavi/Fabijan): p < {umbral}. "
      f"{'¡SRM! investigar el pipeline.' if p_srm < umbral else 'No hay SRM: la razón es compatible con 50/50.'}")

**📖 Cómo se lee.** El chi² da **p ≈ 0.0086**: por encima del **umbral estricto (0,001)**, así que **no se declara SRM** y el experimento es fiable. **⚠️** Con 0,05 se habría declarado una alarma: por eso el umbral de SRM es tan estricto — con muestras muy grandes, desbalances triviales dan p pequeños. Si se activara, la corrección **no** consiste en modificar el número sino **investigar el pipeline** de asignación (`SUPUESTOS_S02.md`, Parte 2.4).

**🔎 Qué hace este código (Diagnóstico 2.3 — tamaño/potencia ex-ante).** Calcula el **n por grupo** requerido para varios lifts objetivo (poder 0.8, α 0.05) y dibuja la curva n vs. efecto mínimo. Es planificación **ex-ante**, no una prueba posterior.

In [ ]:
# Diagnóstico 2.3 - tamaño muestral y potencia EX-ANTE: n requerido según el efecto mínimo (NO escribe)
analysis = NormalIndPower()
p_ref = 0.1902013422818792                          # retención base gate_30 (retention_7)
lifts = np.array([0.01, 0.02, 0.03, 0.05, 0.10])
filas = []
for L in lifts:
    h = abs(proportion_effectsize(p_ref*(1+L), p_ref))
    n_req = analysis.solve_power(effect_size=h, alpha=0.05, power=0.8, ratio=1.0, alternative="two-sided")
    filas.append({"lift_objetivo": f"{L:.0%}", "effect_size_h": round(h, 5), "n_por_grupo": int(np.ceil(n_req))})
print(pd.DataFrame(filas).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.4, 4.2))
Ls = np.linspace(0.01, 0.10, 30)
ns = [analysis.solve_power(effect_size=abs(proportion_effectsize(p_ref*(1+L), p_ref)),
                           alpha=0.05, power=0.8, ratio=1.0) for L in Ls]
ax.plot(Ls*100, ns, color="#c44e52", lw=2)
ax.axvline(2, ls="--", color="gray"); ax.set_yscale("log")
ax.set_xlabel("Lift objetivo (%)"); ax.set_ylabel("n por grupo (escala log)")
ax.set_title("Tamaño muestral requerido vs. efecto mínimo (poder 0.8, alfa 0.05)")
fig.tight_layout()
f_pow = FIG_DIR / "S02_sup_tamano_muestral.png"
fig.savefig(f_pow, dpi=150, bbox_inches="tight"); plt.close(fig)
display(Image(filename=str(f_pow)))

**📖 Cómo se lee.** Detectar un lift del **2 %** exige **~168 000 por grupo**; bajar al 1 % lo **cuadruplica** (la curva crece como $1/h^2$). El n necesario **crece de forma acelerada** cuando el efecto a detectar se reduce: por eso los efectos pequeños requieren un tráfico muy elevado. Dimensionar así **antes** de lanzar es la defensa estructural contra el *peeking* (`SUPUESTOS_S02.md`, Parte 2.3).

**🔎 Qué hace este código (Diagnóstico 2.5 — comparaciones múltiples).** Toma los **dos p-valores** del A/B (`retention_1` y `retention_7`) desde el Excel y aplica **Bonferroni** y **FDR de Benjamini-Hochberg** para ver qué sobrevive al corregir por multiplicidad.

In [ ]:
# Diagnóstico 2.5 - comparaciones múltiples: Bonferroni y FDR sobre las 2 métricas del A/B (NO escribe)
ab = pd.read_excel(XLSX, sheet_name="ab_retencion")
pvals = ab["p_valor"].to_numpy()
rej_b, p_b, _, _ = multipletests(pvals, alpha=0.05, method="bonferroni")
rej_f, p_f, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")
res = pd.DataFrame({"metrica": ab["metrica"], "p_original": pvals.round(5),
                    "p_Bonferroni": p_b.round(5), "signif_Bonf": rej_b,
                    "p_FDR_BH": p_f.round(5), "signif_FDR": rej_f})
print(res.to_string(index=False))

**📖 Cómo se lee.** Con dos pruebas, el umbral de Bonferroni es α/2 = 0,025: **`retention_7`** (p = 0,0016) **sobrevive** a Bonferroni y FDR, mientras que **`retention_1`** (p = 0,074) **no** en ninguna. **💡** Declarar `retention_7` como **métrica primaria (OEC)** de antemano y corregir por multiplicidad evita adoptar una «ganadora» espuria (`SUPUESTOS_S02.md`, Parte 2.5).

## 2.7 — ¿Cómo se verifica que el valor p no induce a error? Intervalos del A/B y *bootstrap* certificado <a id="anexo-ic-bootstrap"></a>

El entregable exige reportar el efecto **con su intervalo**, y la lectura de la réplica afirma que el efecto se sostiene por **tres vías** (t pareada, Wilcoxon y *bootstrap*). Este anexo **calcula efectivamente** esas piezas y las registra en **tres hojas NUEVAS** del Excel, **sin modificar** las 5 hojas de contrato que escribe la celda 95:

| Bloque | Qué calcula | Hoja nueva |
|---|---|---|
| **A.1** | IC 95 % de la **diferencia** de proporciones (Wald no agrupado) y del ***lift*** por **dos** métodos que **no** dan el mismo número | `ab_intervalos` |
| **A.2** | ***Bootstrap*** de la diferencia de proporciones (B = 10 000): punto, IC percentil y **tabla de estabilidad entre 6 semillas** | `bootstrap_ab` |
| **A.3** | ***Bootstrap*** de la media de las diferencias pareadas de `sleep`: la **tercera vía** de la réplica | `bootstrap_sleep` |
| **A.4** | Escribe las tres hojas y **verifica con `assert`** que las 5 de contrato quedan idénticas | — |

**Semillas declaradas:** **42** (la del cuaderno; es la fila que certifica el material de referencia de la sesión) y **7, 123, 2024, 2026, 31415**, solo para medir estabilidad. Todo el remuestreo usa generadores **locales** (`np.random.default_rng(semilla)`), de modo que **no consume** el `rng` global de la celda 5 ni altera ninguna salida anterior.

**⚠️** Un resultado de *bootstrap* **no es una constante**: solo es reproducible junto con su **semilla**, su **número de remuestreos** y la **versión de la librería**. Por eso las tres hojas registran esas columnas y el validador las recomputa desde la base.

**🔎 Qué hace este código.** Calcula, para `retention_7` y `retention_1`, el **IC 95 % de la diferencia** de proporciones $\hat p_1-\hat p_0$ con el error estándar **no agrupado** $\sqrt{\hat p_0(1-\hat p_0)/n_0+\hat p_1(1-\hat p_1)/n_1}$ (el agrupado se reserva para el estadístico $z$ bajo $H_0$), y el **IC del *lift*** por **dos métodos distintos**, que se registran **con rótulo separado** porque no dan el mismo número:

- **(a) Escalado:** se divide el IC de la diferencia entre $\hat p_0$, es decir se trata la base $\hat p_0$ como una **constante conocida**. Es el más simple y el que suele aparecer en los tableros.
- **(b) Delta-método:** sobre el cociente $R=\hat p_1/\hat p_0$, con $\operatorname{Var}(R)\approx R^2\left[\operatorname{Var}(\hat p_1)/\hat p_1^{2}+\operatorname{Var}(\hat p_0)/\hat p_0^{2}\right]$. Propaga **también** la incertidumbre del **denominador**, que el escalado ignora.

El *lift* puntual es idéntico en ambos; **los extremos del intervalo no**.

In [ ]:
# Anexo A.1 - IC 95% de la DIFERENCIA de proporciones y del LIFT por DOS métodos (rotulados)
Z95 = float(stats.norm.ppf(0.975))          # 1.959964


def intervalos_ab(metrica: str) -> list:
    """IC 95% de la diferencia (Wald no agrupado) y del lift por dos métodos distintos."""
    g = cc.groupby("version")[metrica].agg(["sum", "count"])
    x0, n0 = int(g.loc["gate_30", "sum"]), int(g.loc["gate_30", "count"])    # control
    x1, n1 = int(g.loc["gate_40", "sum"]), int(g.loc["gate_40", "count"])    # tratamiento
    p0, p1 = x0 / n0, x1 / n1
    dif = p1 - p0
    ee_dif = np.sqrt(p0 * (1 - p0) / n0 + p1 * (1 - p1) / n1)   # EE NO agrupado (para el IC)
    ic_dif = (dif - Z95 * ee_dif, dif + Z95 * ee_dif)
    lift = dif / p0
    # (a) ESCALADO: el IC de la diferencia dividido por p0 (p0 tratado como constante conocida)
    ic_esc = (ic_dif[0] / p0, ic_dif[1] / p0)
    # (b) DELTA-METODO sobre R = p1/p0: propaga TAMBIEN la varianza del denominador
    R = p1 / p0
    ee_R = R * np.sqrt((p1 * (1 - p1) / n1) / p1**2 + (p0 * (1 - p0) / n0) / p0**2)
    ic_delta = (R - Z95 * ee_R - 1, R + Z95 * ee_R - 1)
    return [
        {"metrica": metrica, "estimador": "diferencia", "metodo": "wald_no_agrupado",
         "punto": dif * 100, "ic95_inf": ic_dif[0] * 100, "ic95_sup": ic_dif[1] * 100, "unidad": "pp"},
        {"metrica": metrica, "estimador": "lift", "metodo": "escalado_ic_diferencia_entre_p0",
         "punto": lift * 100, "ic95_inf": ic_esc[0] * 100, "ic95_sup": ic_esc[1] * 100, "unidad": "%"},
        {"metrica": metrica, "estimador": "lift", "metodo": "delta_metodo",
         "punto": lift * 100, "ic95_inf": ic_delta[0] * 100, "ic95_sup": ic_delta[1] * 100, "unidad": "%"},
    ]


ab_intervalos = pd.DataFrame(intervalos_ab("retention_7") + intervalos_ab("retention_1"))
print(ab_intervalos.round(4).to_string(index=False))

# Los dos IC del lift de retention_7 son DISTINTOS y así se publican, cada uno con su rótulo.
_e = ab_intervalos.loc[1]      # lift retention_7, escalado
_d = ab_intervalos.loc[2]      # lift retention_7, delta-método
print(f"\nretention_7 — diferencia = {ab_intervalos.loc[0, 'punto']:.4f} pp  "
      f"IC95 ({ab_intervalos.loc[0, 'ic95_inf']:.4f}, {ab_intervalos.loc[0, 'ic95_sup']:.4f}) pp")
print(f"retention_7 — lift = {_e['punto']:.4f} %   IC95 escalado      "
      f"({_e['ic95_inf']:.4f}, {_e['ic95_sup']:.4f}) %  -> se redondea a (-7.0 %, -1.6 %)")
print(f"retention_7 — lift = {_d['punto']:.4f} %   IC95 delta-método  "
      f"({_d['ic95_inf']:.4f}, {_d['ic95_sup']:.4f}) %  -> se redondea a (-6.9 %, -1.7 %)")

assert (round(_e["ic95_inf"], 1), round(_e["ic95_sup"], 1)) == (-7.0, -1.6), "IC escalado del lift"
assert (round(_d["ic95_inf"], 1), round(_d["ic95_sup"], 1)) == (-6.9, -1.7), "IC delta-método del lift"
assert _e["ic95_inf"] < _d["ic95_inf"] and _e["ic95_sup"] > _d["ic95_sup"], \
    "el escalado debe ser más ancho que el delta-método en este caso"
assert ab_intervalos.loc[0, "ic95_sup"] < 0 and ab_intervalos.loc[3, "ic95_sup"] > 0, \
    "retention_7 excluye el 0; retention_1 lo cruza"
print("\nOK: ambos IC del lift calculados y rotulados por separado; retention_1 cruza el 0.")

**📖 Cómo se lee.** En `retention_7`, la diferencia es **−0.8201 pp** con IC 95 % **(−1.3282; −0.3121) pp**: el intervalo entero cae por debajo de 0, así que la caída es real y no ruido. El ***lift*** puntual es **−4.3119 %** en ambos métodos, pero su intervalo **no**: el **escalado** da **(−6.98 %; −1.64 %)** → **(−7.0 %; −1.6 %)** y el **delta-método** da **(−6.92 %; −1.70 %)** → **(−6.9 %; −1.7 %)**. **⚠️** La diferencia (≈0.06 pp en cada extremo) no cambia ninguna decisión, pero **sí cambia el rótulo**: quien publique (−7.0 %; −1.6 %) debe llamarlo *IC de la diferencia escalado por la base*, no «delta-método». La explicación completa está en la guía de interpretación de resultados, «Sección 3». En `retention_1` el IC de la diferencia **cruza 0**, coherente con su p = 0.074.

**🔎 Qué hace este código.** Calcula el ***bootstrap* de la diferencia de proporciones** de `retention_7` (tratamiento − control) con **B = 10 000** remuestreos y **6 semillas declaradas**, y para cada una registra la **media de la distribución bootstrap** y su **IC percentil (2.5–97.5)**, tanto en puntos porcentuales como en *lift*. Remuestrear con reemplazo los $n$ indicadores 0/1 de un brazo y contar los éxitos es **exactamente** extraer de una $\mathrm{Binomial}(n,\hat p)$, así que se usa esa vía (idéntica en distribución y sin recorrer 90 189 filas 10 000 veces). La primera fila (**semilla 42**) es la que certifica el validador; las otras cinco existen para **mostrar cuánto se mueve el resultado con la semilla**.

In [ ]:
# Anexo A.2 - bootstrap de la DIFERENCIA de proporciones (retention_7): 6 semillas declaradas
SEMILLAS_BOOT = (42, 7, 123, 2024, 2026, 31415)   # 42 = fila certificada por el material de referencia de la sesión
B_BOOT = 10_000

_g7 = cc.groupby("version")["retention_7"].agg(["sum", "count"])
x0_b, n0_b = int(_g7.loc["gate_30", "sum"]), int(_g7.loc["gate_30", "count"])
x1_b, n1_b = int(_g7.loc["gate_40", "sum"]), int(_g7.loc["gate_40", "count"])
p0_b, p1_b = x0_b / n0_b, x1_b / n1_b
dif_obs_pp = (p1_b - p0_b) * 100

filas_boot = []
for semilla in SEMILLAS_BOOT:
    r = np.random.default_rng(semilla)        # generador LOCAL: no toca el rng global de la celda 5
    b0 = r.binomial(n0_b, p0_b, B_BOOT) / n0_b
    b1 = r.binomial(n1_b, p1_b, B_BOOT) / n1_b
    dif_b = (b1 - b0) * 100                   # en puntos porcentuales
    lift_b = (b1 - b0) / b0 * 100             # en %
    ic_d = np.percentile(dif_b, [2.5, 97.5])
    ic_l = np.percentile(lift_b, [2.5, 97.5])
    filas_boot.append({"semilla": semilla, "B_remuestreos": B_BOOT, "metrica": "retention_7",
                       "dif_observada_pp": dif_obs_pp, "media_boot_pp": float(dif_b.mean()),
                       "ic95_inf_pp": float(ic_d[0]), "ic95_sup_pp": float(ic_d[1]),
                       "lift_media_boot_pct": float(lift_b.mean()),
                       "lift_ic95_inf_pct": float(ic_l[0]), "lift_ic95_sup_pct": float(ic_l[1])})

bootstrap_ab = pd.DataFrame(filas_boot)
print(bootstrap_ab.round(4).to_string(index=False))

_f0 = bootstrap_ab.iloc[0]
_rango = (bootstrap_ab["media_boot_pp"].min(), bootstrap_ab["media_boot_pp"].max())
print(f"\nDiferencia observada (no depende de la semilla): {dif_obs_pp:.4f} pp")
print(f"Bootstrap semilla 42, B={B_BOOT:,}: media {_f0['media_boot_pp']:.4f} pp   "
      f"IC95 percentil ({_f0['ic95_inf_pp']:.4f}, {_f0['ic95_sup_pp']:.4f}) pp")
print(f"Estabilidad entre {len(SEMILLAS_BOOT)} semillas: la media bootstrap cae en "
      f"[{_rango[0]:.4f}; {_rango[1]:.4f}] pp (amplitud {_rango[1] - _rango[0]:.4f} pp)")

assert _rango[1] - _rango[0] < 0.02, "la media bootstrap no debería moverse >0.02 pp entre semillas"
assert all(bootstrap_ab["ic95_sup_pp"] < 0), "todas las semillas deben dar un IC íntegramente negativo"
assert abs(_f0["media_boot_pp"] - dif_obs_pp) < 0.01, "la media bootstrap debe rodear a la observada"
print("OK: el bootstrap de la diferencia existe, es reproducible por semilla y su IC excluye el 0.")

**📖 Cómo se lee.** La diferencia observada, **−0.8201 pp**, no depende de la semilla; lo que sí depende es el *bootstrap* que la rodea. Con **semilla 42** y **B = 10 000**: media **−0.8213 pp**, IC percentil **(−1.3356; −0.3111) pp** — coincide con el IC analítico de A.1 hasta la primera décima de punto porcentual, que es la comprobación cruzada. Entre las **6 semillas** la media se mueve en **[−0.8213; −0.8150] pp**: **6 milésimas de punto porcentual** de ruido de remuestreo. **⚠️** De ahí la regla: un titular de *bootstrap* se publica **con su semilla y su B**, y nunca se cita con más precisión de la que la propia semilla sostiene. Los seis IC excluyen el 0, así que la conclusión (`gate_40` reduce la retención a 7 días) **no** depende del azar del remuestreo.

**🔎 Qué hace este código.** Aplica el ***bootstrap* a la media de las 10 diferencias pareadas** de `sleep` ($d_i=\text{extra}_2-\text{extra}_1$): remuestrea con reemplazo los 10 pares (`rng.integers` sobre los índices), calcula la media de cada remuestreo y toma los percentiles 2.5–97.5. Es la **tercera vía** de la réplica —junto a la t pareada y a Wilcoxon— y la única que **no supone normalidad ni fórmula cerrada**, precisamente lo que se requiere con n = 10 y un Shapiro que rechaza. Se repite con las **6 semillas declaradas** para exhibir su estabilidad.

In [ ]:
# Anexo A.3 - bootstrap de la MEDIA DE LAS DIFERENCIAS PAREADAS de sleep (tercera vía de la réplica)
filas_bs = []
for semilla in SEMILLAS_BOOT:
    r = np.random.default_rng(semilla)                     # generador LOCAL
    idx = r.integers(0, d.size, size=(B_BOOT, d.size))     # remuestreo con reemplazo de los 10 pares
    medias_b = d[idx].mean(axis=1)
    ic_b = np.percentile(medias_b, [2.5, 97.5])
    filas_bs.append({"semilla": semilla, "B_remuestreos": B_BOOT,
                     "media_observada_h": float(d.mean()), "media_boot_h": float(medias_b.mean()),
                     "ic95_inf_h": float(ic_b[0]), "ic95_sup_h": float(ic_b[1])})

bootstrap_sleep = pd.DataFrame(filas_bs)
print(bootstrap_sleep.round(4).to_string(index=False))

_s0 = bootstrap_sleep.iloc[0]
_w_stat, _w_p = stats.wilcoxon(g2, g1)
print(f"\nLas TRES vías sobre el mismo efecto (diferencia media observada = {d.mean():.4f} h):")
print(f"  1) t pareada      : t = {t_pareado:.4f}, p = {p_valor:.6f}, "
      f"IC95 ({ic_inf:.4f}, {ic_sup:.4f}) h")
print(f"  2) Wilcoxon       : estadístico = {_w_stat:.1f}, p = {_w_p:.6f} (sin suponer normalidad)")
print(f"  3) bootstrap (42) : media {_s0['media_boot_h']:.4f} h, "
      f"IC95 percentil ({_s0['ic95_inf_h']:.4f}, {_s0['ic95_sup_h']:.4f}) h, B = {B_BOOT:,}")

assert all(bootstrap_sleep["ic95_inf_h"] > 0), "el IC bootstrap debe excluir el 0 en las 6 semillas"
assert abs(_s0["media_boot_h"] - d.mean()) < 0.02, "la media bootstrap debe rodear a la observada"
assert _w_p < 0.05 and p_valor < 0.05, "las otras dos vías también deben ser significativas"
print("\nOK: las tres vías coinciden en signo y significancia -> la afirmación de «tres vías» es cierta.")

**📖 Cómo se lee.** Con semilla 42 y B = 10 000, el *bootstrap* da una media de **1.5779 h** e IC percentil **(0.950; 2.380) h**: **no incluye 0**, igual que el IC clásico **(0.700; 2.460) h** de la t pareada y que el Wilcoxon (p = 0.0039). Las **tres vías** —paramétrica, de rangos y de remuestreo— apuntan al mismo lado, de modo que la conclusión no descansa en el supuesto de normalidad que Shapiro cuestiona (W = 0.83, p = 0.033). **💡** El intervalo del *bootstrap* es **más estrecho** que el clásico porque los percentiles no arrastran las colas gruesas de la t con gl = 9; con n = 10 conviene reportar **ambos** y adoptar el más conservador. Entre las 6 semillas el extremo inferior se mueve entre 0.95 y 0.96 h, y el superior entre 2.36 y 2.40 h.

**🔎 Qué hace este código.** Escribe las **tres hojas nuevas** (`ab_intervalos`, `bootstrap_ab`, `bootstrap_sleep`) en `resultados/S02_resultados.xlsx` en modo ***append***, sin reabrir ni reescribir las 5 hojas de contrato, y a continuación **relee el archivo desde disco** para verificar con `assert` que (a) las 5 hojas de contrato siguen ahí con **exactamente** los mismos valores que se acaban de escribir en la celda 95, y (b) las 3 hojas nuevas están completas. Es la garantía mecánica de que el anexo **añade** registro sin alterar el contrato.

In [ ]:
# Anexo A.4 - escribir las 3 hojas NUEVAS sin tocar las 5 de contrato (y probarlo con assert)
HOJAS_CONTRATO = ["prueba_t_pareada", "descriptivos", "supuestos", "ab_retencion", "ab_potencia"]
HOJAS_NUEVAS = {"ab_intervalos": ab_intervalos,
                "bootstrap_ab": bootstrap_ab,
                "bootstrap_sleep": bootstrap_sleep}

antes = {h: pd.read_excel(XLSX, sheet_name=h) for h in HOJAS_CONTRATO}   # foto previa (desde disco)

with pd.ExcelWriter(XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    for nombre, tabla in HOJAS_NUEVAS.items():
        tabla.to_excel(xw, sheet_name=nombre, index=False)

despues = {h: pd.read_excel(XLSX, sheet_name=h) for h in HOJAS_CONTRATO}
for h in HOJAS_CONTRATO:
    pd.testing.assert_frame_equal(antes[h], despues[h], check_exact=True,
                                  obj=f"hoja de contrato '{h}'")
for nombre, tabla in HOJAS_NUEVAS.items():
    leida = pd.read_excel(XLSX, sheet_name=nombre)
    assert leida.shape == tabla.shape, f"hoja nueva '{nombre}' incompleta"

from openpyxl import load_workbook as _lw
print("Hojas del Excel tras el anexo:", _lw(XLSX, read_only=True).sheetnames)
print(f"  contrato (intactas, verificadas valor a valor): {', '.join(HOJAS_CONTRATO)}")
print(f"  nuevas de este anexo: {', '.join(HOJAS_NUEVAS)}")
print("\nOK: 3 hojas nuevas escritas; las 5 hojas de contrato quedan idénticas valor a valor.")

**📖 Cómo se lee.** El Excel pasa de **5 a 8 hojas**. Las cinco de contrato se releen desde disco y se comparan **valor a valor** con la foto tomada antes de escribir: si el anexo modificara una sola celda, `assert_frame_equal` haría fallar el cuaderno. Las tres nuevas quedan disponibles para el entregable (que exige el ***lift* con su IC**) y para el material de referencia de la sesión, que las **recomputa desde las bases** y las cruza con estas celdas. **💡** El patrón —contrato escrito una sola vez, registro adicional en hojas nuevas declaradas— es el que permite ampliar el Excel de una sesión ya cerrada sin alterar lo que el deck y el validador ya leen. El **Anexo A.5** aplica el mismo patrón para llevar al Excel el **χ² del SRM** y la **traducción a negocio**.


### Anexo A.5 — Guardarraíl SRM y traducción a negocio, registrados en el Excel  <a id="anexo-srm-impacto"></a>

**❓ Qué se quiere averiguar.** La caída de `retention_7` es estadísticamente significativa, pero ¿es **lo bastante grande** para que el negocio actúe? ¿Cuántos jugadores y cuánto dinero hay detrás de **0,82 puntos porcentuales**?

- **La decisión concreta:** esta es la pregunta que separa la **significancia estadística** de la **relevancia para el negocio**. Un p de 0.0016 dice que el efecto no es ruido; no dice si merece un cambio de producto. La recomendación de mantener `gate_30` se defiende ante un comité con el **tamaño** del efecto traducido a jugadores y a soles, no con el p-valor.
- **Antes de mirar el resultado:** con decenas de miles de usuarios por rama, una diferencia mínima puede resultar significativa sin un efecto apreciable en el negocio; a la inversa, un efecto grande sobre poca muestra puede no alcanzar significancia y aun así merecer atención. Si al traducir esos 0,82 pp a jugadores el número resultase marginal, la caída sería real pero irrelevante y `gate_40` seguiría en discusión; si resulta material, la recomendación deja de apoyarse en el p-valor y pasa a apoyarse en el tamaño. Queda un riesgo adicional: el resultado depende de la **base** que se multiplique —cohorte completa (90 189) o brazo de tratamiento (45 489)—, y equivocarse ahí duplica el denominador del ROI. El ARPU es un supuesto ilustrativo, no un dato del juego.

**🔎 Qué hace este código.** Escribe **dos hojas nuevas** en `resultados/S02_resultados.xlsx`, también en modo ***append*** y sin modificar las ocho anteriores. (1) La hoja **`srm`** registra el **χ² del reparto** que la celda 118 solo imprimía: los dos conteos, el esperado bajo la razón de diseño 50/50, **los dos sumandos por separado**, el χ², sus grados de libertad, el p-valor y el umbral estricto de SRM. Así la derivación de pizarra del Acto 3 se apoya en un registro verificable y el deck la **lee del Excel** en vez de llevarla como literal en el código. (2) La hoja **`impacto_negocio`** traduce el efecto a jugadores y a dinero **con la base declarada en cada fila**: la regla dura de la sesión es que los ≈740 jugadores se calculan sobre la **cohorte completa (90 189, ambas ramas)** y que el **brazo de tratamiento (45 489)** —≈373 jugadores— es la base del costo del experimento y del ROI; confundirlas duplica el número. El ARPU es un **supuesto ilustrativo de negocio**, no un dato del dataset, y así queda rotulado en la propia hoja.


In [ ]:
# Anexo A.5 - hojas `srm` e `impacto_negocio`: cero cifras cableadas en el deck
# (las 8 hojas anteriores quedan INTACTAS y se prueba valor a valor, como en el Anexo A.4)

# --- (1) hoja `srm`: el chi-cuadrado del reparto, con sus DOS sumandos explícitos ---
_vc = cc["version"].value_counts()
n30_srm, n40_srm = int(_vc["gate_30"]), int(_vc["gate_40"])
tot_srm = n30_srm + n40_srm
esp_srm = tot_srm / 2                                   # esperado por grupo bajo la razón 50/50
sum30 = (n30_srm - esp_srm) ** 2 / esp_srm
sum40 = (n40_srm - esp_srm) ** 2 / esp_srm
chi2_a5, p_a5 = stats.chisquare(f_obs=[n30_srm, n40_srm], f_exp=[esp_srm, esp_srm])
assert abs((sum30 + sum40) - float(chi2_a5)) < 1e-9, "el chi2 a mano debe coincidir con scipy"

srm = pd.DataFrame({
    "concepto": ["n_gate30", "n_gate40", "n_total", "esperado_por_grupo_5050",
                 "sumando_gate30", "sumando_gate40", "chi2", "gl", "p_valor", "umbral_srm"],
    "valor": [n30_srm, n40_srm, tot_srm, esp_srm, sum30, sum40,
              float(chi2_a5), 1, float(p_a5), 0.001],
})
print("Hoja `srm` (derivación del guardarraíl, celda 118 registrada):")
print(srm.to_string(index=False))
print(f"  {n30_srm:,} y {n40_srm:,} contra {esp_srm:,.1f} esperados -> "
      f"chi2 = {sum30:.4f} + {sum40:.4f} = {float(chi2_a5):.4f} (gl=1), p = {float(p_a5):.6f}")
print(f"  p esta {'POR DEBAJO' if float(p_a5) < 0.001 else 'POR ENCIMA'} del umbral 0.001: "
      f"{'hay SRM' if float(p_a5) < 0.001 else 'NO se declara SRM'}.")

# --- (2) hoja `impacto_negocio`: cada cifra con SU base declarada (740 != 373) ---
dif_pp_a5 = float(ab_intervalos.loc[0, "punto"])        # retention_7, diferencia en pp (Anexo A.1)
efecto_a5 = abs(dif_pp_a5) / 100                        # 0.008201... en proporción
jug_cohorte = efecto_a5 * tot_srm                       # BASE: cohorte completa (ambas ramas)
jug_trat = efecto_a5 * n40_srm                          # BASE: brazo de tratamiento (costo/ROI)
ARPU_ILUSTRATIVO = 5.0                                  # S/ por jugador - SUPUESTO de negocio
ingreso_protegido = round(jug_cohorte) * ARPU_ILUSTRATIVO

impacto_negocio = pd.DataFrame({
    "concepto": ["dif_ret7_pp", "n_cohorte_total", "n_brazo_tratamiento",
                 "jugadores_cohorte", "jugadores_brazo_tratamiento",
                 "arpu_ilustrativo_soles", "ingreso_protegido_cohorte_soles"],
    "valor": [dif_pp_a5, tot_srm, n40_srm, jug_cohorte, jug_trat,
              ARPU_ILUSTRATIVO, ingreso_protegido],
    "base_declarada": [
        "retention_7: tratamiento - control (Anexo A.1)",
        "cohorte completa: gate_30 + gate_40 (ambas ramas)",
        "solo gate_40 (brazo de tratamiento) = costo del experimento",
        "cohorte completa de 90 189 (ambas ramas) -- NO es el brazo de tratamiento",
        "brazo de tratamiento de 45 489 -- ESTA es la base del ROI",
        "supuesto de negocio ILUSTRATIVO, no procede del dataset",
        "740 jugadores (redondeo de la cohorte) x ARPU ilustrativo S/ 5",
    ],
    "unidad": ["pp", "jugadores", "jugadores", "jugadores", "jugadores", "S/", "S/"],
})
print("\nHoja `impacto_negocio` (traducción a negocio, cada cifra con su base):")
print(impacto_negocio[["concepto", "valor", "unidad"]].to_string(index=False))

assert round(jug_cohorte) == 740 and round(jug_trat) == 373, "740 (cohorte) y 373 (tratamiento)"
assert abs(jug_cohorte - 2 * jug_trat) < 20, "la cohorte es ~2x el brazo: no se pueden intercambiar"

# --- escritura en append + prueba de que las 8 hojas anteriores no se movieron ---
HOJAS_PREVIAS = ["prueba_t_pareada", "descriptivos", "supuestos", "ab_retencion", "ab_potencia",
                 "ab_intervalos", "bootstrap_ab", "bootstrap_sleep"]
HOJAS_A5 = {"srm": srm, "impacto_negocio": impacto_negocio}

antes_a5 = {h: pd.read_excel(XLSX, sheet_name=h) for h in HOJAS_PREVIAS}
with pd.ExcelWriter(XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    for nombre, tabla in HOJAS_A5.items():
        tabla.to_excel(xw, sheet_name=nombre, index=False)
despues_a5 = {h: pd.read_excel(XLSX, sheet_name=h) for h in HOJAS_PREVIAS}
for h in HOJAS_PREVIAS:
    pd.testing.assert_frame_equal(antes_a5[h], despues_a5[h], check_exact=True,
                                  obj=f"hoja previa '{h}'")
for nombre, tabla in HOJAS_A5.items():
    assert pd.read_excel(XLSX, sheet_name=nombre).shape == tabla.shape, f"hoja '{nombre}' incompleta"

from openpyxl import load_workbook as _lw2
print("\nHojas del Excel tras el Anexo A.5:", _lw2(XLSX, read_only=True).sheetnames)
print("OK: 2 hojas nuevas (`srm`, `impacto_negocio`); las 8 anteriores, idénticas valor a valor.")


**📖 Cómo se lee.** El Excel pasa de **8 a 10 hojas** y con ello el deck alcanza el contrato de **cero literales**: el χ² del SRM (**6.9024**, p = **0.0086**), los **≈740 jugadores por cohorte de 90 189** y los **≈ S/ 3 700** de ingreso protegido dejan de estar escritos como literales en el generador y se **leen del Excel**, igual que ya ocurría con el lift y sus intervalos. La hoja `impacto_negocio` obliga a **declarar la base** de cada cifra en la propia fila, que es la defensa contra el error más costoso de la sesión: multiplicar el efecto por la cohorte completa (90 189 → 740) y luego usar ese número como si fuera el costo del experimento (45 489 → 373), que **duplicaría el denominador del ROI**. **💡** El ARPU de S/ 5 es un **supuesto ilustrativo**, no un dato de Cookie Cats: la hoja lo dice en su columna `base_declarada` para que no se cite como si fuera una medición.


## Práctica — Drills (ejercicios) (Sección 9 del cuaderno)

Enunciados para practicar. Las soluciones se trabajan en clase; el desarrollo completo se entrega según `evaluacion/drills.docx`.

1. **Tamaño muestral.** Calcular el tamaño muestral por grupo necesario para detectar un **lift del 2%** con **poder 0.8** (y $\alpha=0.05$) sobre una métrica de proporción. Comparar el resultado con el tráfico disponible y estimar la duración del test.
2. **Elegir la prueba.** Para cada escenario, indicar la prueba adecuada y justificar: (a) comparar la media de **2 grupos** independientes; (b) comparar **más de 2 grupos**; (c) asociar dos variables **categóricas**; (d) comparar dos grupos cuando la variable **no es normal**.
3. **Interpretar un p-valor.** Un test arroja **p = 0.04** con **n = 50 000**. Discutir la diferencia entre significancia **estadística** y **práctica**: ¿qué información adicional (tamaño del efecto, IC, lift) se necesita antes de recomendar el cambio?

## 2.9 — ¿Qué no se puede afirmar, y qué sigue en S03? Cierre (Sección 10 del cuaderno)

### Entregable de la sesión
Analizar el A/B test real de *Cookie Cats* y emitir una **recomendación de negocio documentada** (hipótesis, tamaño muestral y poder, p-valor, *lift*, riesgo de *peeking* y decisión). El enunciado y la **rúbrica vigesimal (0–20)** están en `evaluacion/entregable.docx`; el trabajo se apoya en `laboratorio/GUIA_LABORATORIO_S02.docx`, el checklist `plantillas/reporte_eda_checklist.docx` y la calculadora `plantillas/calculadora_tamano_muestral_ab.ipynb`. Modalidad: parejas, entrega individual.

### Control corto
La sesión cierra con un **control corto** de opción y desarrollo sobre EDA, elección de prueba, lectura del p-valor y diseño de un A/B (banco de ítems `[S02]`).

### Vínculo con el proyecto integrador
El **EDA y la calidad de datos** de esta sesión alimentan la fase de *Comprensión y preparación de los datos* del **proyecto integrador**; el **A/B testing** es la puerta a la **inferencia causal** (S12): el experimento aleatorizado es el estándar de oro para atribuir causa. La distinción **significancia estadística vs. práctica** y la lectura del **p-valor** se reutilizan en toda prueba de hipótesis del curso.

### Materiales de apoyo de esta sesión
- Mapa celda↔slide para el deck: el cuaderno de la sesión.
- Supuestos (fuente canónica): la guía de supuestos de la sesión.
- Validación de la réplica y de los intervalos/bootstrap (recomputa desde la base): el material de referencia de la sesión.
- Guía de laboratorio, plantillas y evaluación: `laboratorio/`, `plantillas/`, `evaluacion/`.

### Para seguir explorando
Lecturas y casos recientes de A/B testing y experimentación (detalle y enlaces verificados en las fuentes de actualidad de la sesión):

- **AgentA/B: Automated and Scalable Web A/B Testing with Interactive LLM Agents** — arXiv, 13/04/2025. Simular usuarios con agentes LLM para prototipar experimentos antes del tráfico real.
- **Harnessing the Power of Interleaving and Counterfactual Evaluation for Airbnb Search Ranking** — arXiv (Airbnb), 01/08/2025. Técnicas que multiplican la sensibilidad para descartar ideas antes del A/B completo.
- **Beyond Normality: Reliable A/B Testing with Non-Gaussian Data** — arXiv, 26/10/2025. Por qué las métricas sesgadas invalidan la t estándar y cómo corregirlo.
- **An Investigation of p-Hacking in E-Commerce A/B Testing** — *Information Systems Research* 36(3), 01/2025. Auditoría empírica de la disciplina experimental en más de 2 000 tests.
- **Scaling A/B Tests Like Netflix & Microsoft** — Venue, 14/12/2025. La experimentación como cultura: plataforma, métricas estandarizadas y guardarraíles.

---
*Fin del cuaderno S02. Todos los resultados numéricos se recalcularon aquí y se exportaron a `resultados/S02_resultados.xlsx` (5 hojas de contrato + 3 hojas del **Anexo A**: `ab_intervalos`, `bootstrap_ab`, `bootstrap_sleep`); las figuras de resultados se generaron leyendo ese Excel. Los bloques 🖐️/✅/🧮/🧱 y la «Sección 8» (supuestos) no escriben en el Excel de contrato, y el Anexo A solo añade hojas nuevas.*